# Analysis Models And Sensitivity Analyses

Purpose: fit the final logistic regression models, sensitivity analyses, odds-ratio tables, model-comparison summaries, and Figure 1 interaction panels.

This notebook uses the unified analysis dataset created in `05_build_exposure_analysis_dataset_clean.ipynb`. The same thread-level file contains combined-provider exposures and separate Physician/APP and other-human-HCW exposures. Models use patient reply within 7 days as the primary outcome, clustered standard errors by patient, month-year fixed effects, and the 7-day encounter control where specified.

## Scope

This notebook does:

- load `analysis_df_all_exposures_with_encounter_covariates.parquet`
- prepare covariates and apply the analysis date window
- compare original adjusted and encounter-adjusted model specifications
- fit primary encounter-adjusted provider/system models
- prepare focused Physician/APP and other-HCW subsets from the unified dataset
- fit focused Phys/APP-initiated and other-HCW-initiated models
- export odds-ratio tables, model comparison summaries, and Figure 1 panels

This notebook does not:

- rebuild message preprocessing, thread IDs, demographics/CCI, or encounter timelines
- create row-level examples
- display message text or patient identifiers

## Downstream Outputs

Outputs include model summaries, odds-ratio tables, and Figure 1 files saved by the notebook. Row-level analysis datasets remain restricted EHR-derived data.


## 1. Load Primary Analysis Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORKING_DATA_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / "runs" / f"analysis_models_{RUN_ID}"
TABLE_DIR = RUN_DIR / "tables"
FIGURE_DIR = RUN_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR.mkdir(parents=True, exist_ok=False)

UNIFIED_ANALYSIS_PATH = WORKING_DATA_DIR / "analysis_df_all_exposures_with_encounter_covariates.parquet"
if not UNIFIED_ANALYSIS_PATH.exists():
    raise FileNotFoundError(
        f"Missing unified analysis dataset: {UNIFIED_ANALYSIS_PATH}\n"
        "Run 05_build_exposure_analysis_dataset_clean.ipynb first."
    )

PRIMARY_ANALYSIS_PATH = UNIFIED_ANALYSIS_PATH
FOCUSED_ANALYSIS_PATH = UNIFIED_ANALYSIS_PATH
analysis_df = pd.read_parquet(PRIMARY_ANALYSIS_PATH, engine="pyarrow")

print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_DATA_DIR:", WORKING_DATA_DIR)
print("RUN_DIR:", RUN_DIR)
print("PRIMARY_ANALYSIS_PATH:", PRIMARY_ANALYSIS_PATH)
print("FOCUSED_ANALYSIS_PATH:", FOCUSED_ANALYSIS_PATH)

## 2. Prepare Covariates And Apply Analysis Window

In [ ]:
analysis_df["Age65"] = pd.Categorical(
    analysis_df["Age65"],
    categories=["<65", "≥65"],
    ordered=False
)

analysis_df["Race"] = pd.Categorical(
    analysis_df["Race"],
    categories=["White", "Black/African American", "Middle Eastern/North African", "Asian", "Native American/Alaskan", "Pacific Islander", "Other/Multiple/Unknown"],
    ordered=False
)

analysis_df["Ethnicity"] = pd.Categorical(
    analysis_df["Ethnicity"],
    categories=["Not Hispanic/Latino", "Hispanic/Latino", "Unknown/Other"],
    ordered=False
)

# Convert pandas StringDtype to a Patsy-compatible categorical dtype.
analysis_df["Gender"] = analysis_df["Gender"].astype("category")

# Ensure Charlson_group is categorical
analysis_df["Charlson_group"] = analysis_df["Charlson_group"].astype("category")

# Ensure CREATED_TIME is datetime
analysis_df["thread_first_time"] = pd.to_datetime(analysis_df["thread_first_time"])
# Create month-year variable for time fixed effects
analysis_df["month_year"] = analysis_df["thread_first_time"].dt.to_period("M").astype(str)

In [ ]:
# Collabse Race
import numpy as np

analysis_df["Race"] = np.select(
    [
        analysis_df["Race"].eq("White"),
        analysis_df["Race"].eq("Black/African American"),
        analysis_df["Race"].eq("Asian"),
    ],
    [
        "White",
        "Black/African American",
        "Asian",
    ],
    default="Other/Multiple/Unknown"
)

# (optional) make it a categorical with White as the reference
analysis_df["Race"] = pd.Categorical(
    analysis_df["Race"],
    categories=["White", "Black/African American", "Asian", "Other/Multiple/Unknown"],
    ordered=False
)

# sanity check
tab = analysis_df["Race"].value_counts(dropna=False).to_frame("n")
tab["%"] = (tab["n"] / tab["n"].sum() * 100).round(1)
print(tab)

In [ ]:
import pandas as pd

df = analysis_df.copy()
# Filter out rows with missing Age65
df1 = df.dropna(subset=['Age65']).copy()

print(f"Rows before filtering: {len(df):,}")
print(f"Rows after filtering: {len(df1):,}")
print(f"Rows removed: {len(df) - len(df1):,}")

# Exclude the early and last 2 weeks


# Define date boundaries
start_date = pd.Timestamp("2021-03-01")
end_date = pd.Timestamp("2023-12-31")

# Compute cutoff windows
start_cutoff = start_date + pd.Timedelta(days=30)   # exclude early 2 weeks
end_cutoff = end_date - pd.Timedelta(days=30)       # exclude last 2 weeks

# Filter threads within valid range
filtered_df = df1[
    (df1["thread_first_time"] >= start_cutoff) &
    (df1["thread_first_time"] <= end_cutoff)
].copy()

print(f"✅ Kept {len(filtered_df):,} rows out of {len(df1):,}")
print(f"   Data range kept: {filtered_df['thread_first_time'].min()} → {filtered_df['thread_first_time'].max()}")
print("   (Removed threads from the first and last 30 days of data).")

## 3. Descriptive Summaries

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================
# 1️⃣ Collapse to patient level
# Each patient appears once, using their first message record
# ==============================================================
patient_df = (
    filtered_df.sort_values("thread_first_time")  # or equivalent timestamp
    .drop_duplicates(subset="DMRN", keep="first")  # 1 row per patient
)
print(f"Unique patients: {len(patient_df):,}")

# ==============================================================
# 2️⃣ Median [IQR] Age
# ==============================================================
median_age = patient_df['Age_at_Message'].median()
q1, q3 = patient_df['Age_at_Message'].quantile([0.25, 0.75])
print(f"Median age: {median_age:.1f} [IQR {q1:.1f}–{q3:.1f}]")

# ==============================================================
# 3️⃣ Gender distribution (n, %)
# ==============================================================
gender_counts = patient_df['Gender'].value_counts(dropna=False)
gender_pct = (gender_counts / len(patient_df) * 100).round(1)
gender_summary = pd.DataFrame({'n': gender_counts, '%': gender_pct})
print("\nGender Distribution:")
print(gender_summary)

# ==============================================================
# 4️⃣ Race distribution (n, %)
# ==============================================================
race_counts = patient_df['Race'].value_counts(dropna=False)
race_pct = (race_counts / len(patient_df) * 100).round(1)
race_summary = pd.DataFrame({'n': race_counts, '%': race_pct})
print("\nRace Distribution:")
print(race_summary)

# ==============================================================
# 5️⃣ Ethnicity distribution (n, %)
# ==============================================================
eth_counts = patient_df['Ethnicity'].value_counts(dropna=False)
eth_pct = (eth_counts / len(patient_df) * 100).round(1)
eth_summary = pd.DataFrame({'n': eth_counts, '%': eth_pct})
print("\nEthnicity Distribution:")
print(eth_summary)

# ==============================================================
# 6️⃣ Charlson Index groups (0, 1–2, 3–4, ≥5)
# ==============================================================
charlson_counts = patient_df['Charlson_group'].value_counts().sort_index()
charlson_pct = (charlson_counts / len(patient_df) * 100).round(1)
charlson_summary = pd.DataFrame({'n': charlson_counts, '%': charlson_pct})
print("\nCharlson Index (grouped):")
print(charlson_summary)

In [ ]:
import pandas as pd
import numpy as np

df = filtered_df.copy()

# ---------------------------
# Helper formatters
# ---------------------------
def n_pct(series):
    n = int(series.notna().sum())
    k = int(series.fillna(0).astype(int).sum())
    pct = (k / n * 100) if n else np.nan
    return f"{k:,} ({pct:.1f}%)", n

def mean_sd(series, decimals=2):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return ""
    return f"{s.mean():.{decimals}f} ({s.std(ddof=1):.{decimals}f})"

def median_iqr(series, decimals=2):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return ""
    q1, med, q3 = np.percentile(s, [25, 50, 75])
    return f"{med:.{decimals}f} ({q1:.{decimals}f}–{q3:.{decimals}f})"

def min_max(series, decimals=0):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return ""
    return f"{s.min():.{decimals}f}–{s.max():.{decimals}f}"

# ---------------------------
# Pick variables
# ---------------------------
outcomes = ["patient_replied_7d", "patient_replied_14d", "patient_replied_30d", "patient_replied_ever"]

windows = [7, 14, 30, 90]
exposures = []
for wd in windows:
    exposures += [
        f"provider_msgs_past_{wd}d",
        f"system_msgs_past_{wd}d",
        f"patient_initiated_threads_past_{wd}d",
    ]

# Encounter covariates
encounter_windows = [7, 30]

for wd in encounter_windows:
    exposures += [
        f"encounters_past_{wd}d",
    ]

# Keep only those that exist
outcomes = [c for c in outcomes if c in df.columns]
exposures = [c for c in exposures if c in df.columns]

# ---------------------------
# Build table
# ---------------------------
rows = []

# Outcomes: n (%)
for c in outcomes:
    val, denom = n_pct(df[c])
    rows.append({
        "Variable": c,
        "Type": "Outcome (n, %)",
        "Summary": val,
        "N (non-missing)": denom
    })

# Exposures: mean(SD), median(IQR), min–max
for c in exposures:
    rows.append({
        "Variable": c,
        "Type": "Exposure (count)",
        "Mean (SD)": mean_sd(df[c], decimals=2),
        "Median (IQR)": median_iqr(df[c], decimals=2),
        "Min–Max": min_max(df[c], decimals=0),
        "N (non-missing)": int(pd.to_numeric(df[c], errors="coerce").notna().sum())
    })

summary_table = pd.DataFrame(rows)

display(summary_table)

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------
# 0) Build patient-level demographics from message-level df
# ---------------------------
df = filtered_df.copy()

# Ensure datetime
df["thread_first_time"] = pd.to_datetime(df["thread_first_time"], errors="coerce")

demo_cols = [c for c in ["Age_at_Message","Age65","Age3cat","Gender","Race","Ethnicity","Charlson_group"] if c in df.columns]

# one row per patient (use first observed non-missing; ok for mostly time-invariant fields)
patient_df = (
    df.dropna(subset=["DMRN", "thread_first_time"])
      .sort_values(["DMRN", "thread_first_time"])
      .groupby("DMRN", as_index=False)[demo_cols]
      .first()
)

# ---------------------------
# 1) Continuous summary (Age_at_Message only)
# ---------------------------
def cont_summary(s):
    s = pd.to_numeric(s, errors="coerce")
    n = s.notna().sum()
    mean = s.mean()
    sd = s.std()
    med = s.median()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    mn = s.min()
    mx = s.max()
    return pd.Series({
        "N (non-missing)": int(n),
        "Mean (SD)": f"{mean:.2f} ({sd:.2f})" if n else "",
        "Median (IQR)": f"{med:.2f} ({q1:.2f}–{q3:.2f})" if n else "",
        "Min–Max": f"{mn:.2f}–{mx:.2f}" if n else ""
    })

cont_vars = [c for c in ["Age_at_Message"] if c in patient_df.columns]
cont_summary_table = (
    pd.DataFrame({"Variable": cont_vars})
      .assign(**{
          "N (non-missing)": [cont_summary(patient_df[v])["N (non-missing)"] for v in cont_vars],
          "Mean (SD)":       [cont_summary(patient_df[v])["Mean (SD)"]       for v in cont_vars],
          "Median (IQR)":    [cont_summary(patient_df[v])["Median (IQR)"]    for v in cont_vars],
          "Min–Max":         [cont_summary(patient_df[v])["Min–Max"]         for v in cont_vars],
      })
)

# ---------------------------
# 2) Categorical summary (multi-row per variable) — NO categorical mutation
# ---------------------------
def cat_table(df_in, var, include_missing=True):
    s = df_in[var]

    # convert to object safely (avoids "Cannot setitem on a Categorical...")
    s2 = s.astype("object")
    if include_missing:
        s2 = s2.fillna("Missing")
    else:
        s2 = s2.dropna()

    vc = s2.value_counts(dropna=False)
    n = int(vc.sum())
    out = (
        vc.rename("n")
          .reset_index()
          .rename(columns={"index": "Level"})
    )
    out["pct"] = (out["n"] / n * 100).round(1)
    out.insert(0, "Variable", var)
    return out

cat_vars = [c for c in ["Age65","Age3cat","Gender","Race","Ethnicity","Charlson_group"] if c in patient_df.columns]
cat_summary_table = pd.concat([cat_table(patient_df, v, include_missing=True) for v in cat_vars], ignore_index=True)

# ---------------------------
# 3) Display
# ---------------------------
display(cont_summary_table)
display(cat_summary_table)

In [ ]:
import pandas as pd
import numpy as np

# Get thread-level data from filtered_df
# Group by thread_id and get the initiator sender for each thread
thread_initiators = filtered_df.groupby('thread_id')['thread_initiator_sender'].first()

# Count threads by initiator type
initiator_counts = thread_initiators.value_counts()
initiator_pct = (initiator_counts / len(thread_initiators) * 100).round(1)

# Combine into summary table
initiator_summary = pd.DataFrame({
    'Initiator Type': initiator_counts.index,
    'Count': initiator_counts.values,
    'Percentage': initiator_pct.values
})

print("Threads by Initiator Type:")
display(initiator_summary)

# Summary: Physician/APP vs Other staff
physician_app_count = initiator_counts.get('Physician', 0) + initiator_counts.get('APP', 0)
physician_app_pct = (physician_app_count / len(thread_initiators) * 100)

other_staff_count = initiator_counts.sum() - physician_app_count
other_staff_pct = 100 - physician_app_pct

summary_grouped = pd.DataFrame({
    'Group': ['Physician/APP', 'Other Staff (including System/Patient)'],
    'Count': [physician_app_count, other_staff_count],
    'Percentage': [round(physician_app_pct, 1), round(other_staff_pct, 1)]
})

print("\n\nGrouped Summary:")
display(summary_grouped)

In [ ]:
print(filtered_df["thread_initiator_sender"].value_counts(dropna=False))

## 4. Original Adjusted Versus Encounter-Adjusted Model Comparison

For each exposure window, the original adjusted model and encounter-adjusted model are fit on the same complete-case sample.

In [ ]:
# Model Fit Comparison:
# Original vs Encounter-Adjusted Models
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2
from IPython.display import display

# ------------------------------------------------------------
# Compare original adjusted vs encounter-adjusted models
# ------------------------------------------------------------
df_compare = filtered_df.copy()

outcome = "patient_replied_7d"
periods = ["30d", "90d"]

base_controls = ["Ethnicity", "C(Charlson_group)"]
race_prefix = 'C(Race, Treatment(reference="White"))'
age_term = "Age_at_Message"
encounter_term = "log_encounters_past_7d"

df_compare[outcome] = df_compare[outcome].astype(float).fillna(0).astype(int)

if encounter_term not in df_compare.columns:
    if "encounters_past_7d" in df_compare.columns:
        df_compare[encounter_term] = np.log1p(df_compare["encounters_past_7d"])
    else:
        raise ValueError(f"Missing required encounter control: {encounter_term}")

comparison_models = {}
fit_rows = []
coef_rows = []

def mcfadden_r2(model):
    return 1 - (model.llf / model.llnull)

def or_ci_p(model, term):
    if term not in model.params.index:
        return np.nan, np.nan, np.nan, np.nan

    beta = float(model.params[term])
    ci = model.conf_int().loc[term].astype(float)
    p = float(model.pvalues[term])

    return (
        float(np.exp(beta)),
        float(np.exp(ci.iloc[0])),
        float(np.exp(ci.iloc[1])),
        p
    )

for period in periods:
    x_prov  = f"log_provider_msgs_past_{period}"
    x_sys   = f"log_system_msgs_past_{period}"
    x_int   = f"{x_prov}:{x_sys}"
    x_int_alt = f"{x_sys}:{x_prov}"
    x_pinit = f"log_patient_initiated_threads_past_{period}"

    original_terms = (
        [x_prov, x_sys, x_int, x_pinit] +
        [age_term] +
        ["C(Gender)"] +
        [race_prefix] +
        base_controls +
        ["C(month_year)"]
    )

    encounter_terms = (
        [x_prov, x_sys, x_int, x_pinit, encounter_term] +
        [age_term] +
        ["C(Gender)"] +
        [race_prefix] +
        base_controls +
        ["C(month_year)"]
    )

    original_formula = f"{outcome} ~ " + " + ".join(original_terms)
    encounter_formula = f"{outcome} ~ " + " + ".join(encounter_terms)

    union_na_vars = (
        [outcome, "DMRN", "month_year", age_term, "Gender", "Race", encounter_term] +
        base_controls +
        [x_prov, x_sys, x_pinit]
    )
    union_na_vars = [v for v in union_na_vars if v in df_compare.columns]

    df_fit = df_compare.dropna(subset=list(set(union_na_vars))).copy()

    m_original = smf.logit(original_formula, data=df_fit).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_fit["DMRN"]},
        disp=False
    )

    m_encounter = smf.logit(encounter_formula, data=df_fit).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_fit["DMRN"]},
        disp=False
    )

    comparison_models[(period, "original_adjusted")] = m_original
    comparison_models[(period, "encounter_adjusted")] = m_encounter

    lr_stat = 2 * (m_encounter.llf - m_original.llf)
    lr_df = int(m_encounter.df_model - m_original.df_model)
    lr_p = float(chi2.sf(lr_stat, lr_df)) if lr_df > 0 else np.nan

    fit_rows.append({
        "period": period,
        "N": int(df_fit.shape[0]),
        "original_AIC": float(m_original.aic),
        "encounter_AIC": float(m_encounter.aic),
        "delta_AIC_encounter_minus_original": float(m_encounter.aic - m_original.aic),
        "original_BIC": float(m_original.bic),
        "encounter_BIC": float(m_encounter.bic),
        "delta_BIC_encounter_minus_original": float(m_encounter.bic - m_original.bic),
        "original_McFadden_pseudo_R2": float(mcfadden_r2(m_original)),
        "encounter_McFadden_pseudo_R2": float(mcfadden_r2(m_encounter)),
        "delta_McFadden_pseudo_R2": float(mcfadden_r2(m_encounter) - mcfadden_r2(m_original)),
        "LR_statistic": float(lr_stat),
        "LR_df": lr_df,
        "LR_P_value": lr_p,
    })

    coefficient_terms = [
        ("Provider messages", x_prov),
        ("System messages", x_sys),
        ("Provider x System interaction", x_int if x_int in m_original.params.index else x_int_alt),
        ("Patient-initiated threads", x_pinit),
        ("Encounters, past 7 days", encounter_term),
    ]

    for label, term in coefficient_terms:
        orig_or, orig_lo, orig_hi, orig_p = or_ci_p(m_original, term)
        enc_or, enc_lo, enc_hi, enc_p = or_ci_p(m_encounter, term)

        coef_rows.append({
            "period": period,
            "term": label,
            "raw_term": term,
            "original_OR": orig_or,
            "original_95CI_low": orig_lo,
            "original_95CI_high": orig_hi,
            "original_P_value": orig_p,
            "encounter_adjusted_OR": enc_or,
            "encounter_adjusted_95CI_low": enc_lo,
            "encounter_adjusted_95CI_high": enc_hi,
            "encounter_adjusted_P_value": enc_p,
            "OR_percent_change_after_encounter_adjustment": (
                ((enc_or - orig_or) / orig_or) * 100
                if pd.notna(orig_or) and orig_or != 0 and pd.notna(enc_or)
                else np.nan
            ),
        })

model_fit_comparison = pd.DataFrame(fit_rows)
coefficient_comparison = pd.DataFrame(coef_rows)

display(model_fit_comparison)
display(coefficient_comparison)

In [ ]:
# Table S5. Model-fit comparison
# ============================================================

from IPython.display import display, Markdown
import numpy as np
import pandas as pd


required_columns = {
    "period",
    "original_AIC",
    "encounter_AIC",
    "delta_AIC_encounter_minus_original",
    "original_BIC",
    "encounter_BIC",
    "delta_BIC_encounter_minus_original",
    "original_McFadden_pseudo_R2",
    "encounter_McFadden_pseudo_R2",
    "LR_statistic",
    "LR_df",
    "LR_P_value",
}

missing_columns = required_columns - set(model_fit_comparison.columns)
if missing_columns:
    raise KeyError(
        "Run the model-comparison cell first. Missing columns: "
        + ", ".join(sorted(missing_columns))
    )


def format_number(value, decimals=1):
    if pd.isna(value):
        return "\u2014"
    return f"{value:,.{decimals}f}"


def format_r2(value):
    if pd.isna(value):
        return "\u2014"
    return f"{value:.4f}"


def format_p_value(value):
    if pd.isna(value):
        return "\u2014"
    if value < 0.001:
        return "<.001"
    return f"{value:.3f}".lstrip("0")


def format_lr(value, df):
    if pd.isna(value) or pd.isna(df):
        return "\u2014"
    return f"{value:,.1f}; df={int(df)}"


table_s5_rows = []

period_order = ["30d", "90d"]
period_labels = {
    "30d": "30 days",
    "90d": "90 days",
}

fit_lookup = model_fit_comparison.set_index("period")

for period in period_order:
    if period not in fit_lookup.index:
        raise KeyError(f"Missing model-fit results for {period}")

    result = fit_lookup.loc[period]

    # Primary model
    table_s5_rows.append({
        "Message exposure window": period_labels[period],
        "Model specification": "Primary specification",
        "AIC": format_number(result["original_AIC"]),
        "BIC": format_number(result["original_BIC"]),
        "McFadden R\u00b2": format_r2(
            result["original_McFadden_pseudo_R2"]
        ),
        "\u0394AIC": "\u2014",
        "\u0394BIC": "\u2014",
        "LR \u03c7\u00b2 (df)": "\u2014",
        "P value": "\u2014",
    })

    # Encounter-adjusted model
    table_s5_rows.append({
        "Message exposure window": "",
        "Model specification": "Encounter-adjusted specification",
        "AIC": format_number(result["encounter_AIC"]),
        "BIC": format_number(result["encounter_BIC"]),
        "McFadden R\u00b2": format_r2(
            result["encounter_McFadden_pseudo_R2"]
        ),
        "\u0394AIC": format_number(
            result["delta_AIC_encounter_minus_original"]
        ),
        "\u0394BIC": format_number(
            result["delta_BIC_encounter_minus_original"]
        ),
        "LR \u03c7\u00b2 (df)": format_lr(
            result["LR_statistic"],
            result["LR_df"],
        ),
        "P value": format_p_value(result["LR_P_value"]),
    })


table_s5 = pd.DataFrame(table_s5_rows)

table_s5_title = (
    "Table S5. Comparison of model fit between the primary and "
    "encounter-adjusted model specifications."
)

table_s5_note = (
    "Note. Primary and encounter-adjusted models were estimated on the "
    "same complete-case sample within each exposure window. The "
    "encounter-adjusted specification additionally included the number "
    "of encounters in the preceding 7 days, transformed using "
    "log(1+count). Delta AIC and Delta BIC represent the encounter-adjusted "
    "value minus the primary-specification value. The likelihood-ratio "
    "test compares the nested model specifications. AIC indicates Akaike "
    "information criterion; BIC, Bayesian information criterion; LR, "
    "likelihood ratio."
)

display(Markdown(f"**{table_s5_title}**"))
display(table_s5)
display(Markdown(table_s5_note))


# ------------------------------------------------------------
# Save the formatted table and note
# ------------------------------------------------------------

table_s5_path = TABLE_DIR / "Table_S5_model_fit_comparison.csv"
table_s5_note_path = TABLE_DIR / "Table_S5_model_fit_comparison_note.txt"

table_s5.to_csv(
    table_s5_path,
    index=False,
    encoding="utf-8-sig",
)

table_s5_note_path.write_text(
    table_s5_title + "\n\n" + table_s5_note,
    encoding="utf-8",
)

print("Saved Table S5:")
print(" -", table_s5_path)
print(" -", table_s5_note_path)

In [ ]:
# ============================================================
# Table S6. Primary vs encounter-adjusted model estimates
# ============================================================

from IPython.display import display, Markdown
import numpy as np
import pandas as pd


required_columns = {
    "period",
    "term",
    "original_OR",
    "original_95CI_low",
    "original_95CI_high",
    "encounter_adjusted_OR",
    "encounter_adjusted_95CI_low",
    "encounter_adjusted_95CI_high",
    "OR_percent_change_after_encounter_adjustment",
}

missing_columns = required_columns - set(coefficient_comparison.columns)
if missing_columns:
    raise KeyError(
        "Run the model-comparison cell first. Missing columns: "
        + ", ".join(sorted(missing_columns))
    )


def format_or_ci(or_value, ci_low, ci_high, decimals=2):
    values = [or_value, ci_low, ci_high]
    if any(pd.isna(value) for value in values):
        return "\u2014"

    return (
        f"{or_value:.{decimals}f} "
        f"({ci_low:.{decimals}f}-{ci_high:.{decimals}f})"
    )


def format_relative_change(value):
    if pd.isna(value):
        return "\u2014"
    return f"{value:.2f}%"


period_order = ["30d", "90d"]
period_labels = {
    "30d": "30 days",
    "90d": "90 days",
}

term_order = [
    "Provider messages",
    "System messages",
    "Provider x System interaction",
    "Patient-initiated threads",
    "Encounters, past 7 days",
]


def display_variable_name(term, period):
    label_map = {
        "Provider messages": "HCW messages",
        "System messages": "System messages",
        "Provider x System interaction": "HCW x System interaction",
        "Patient-initiated threads": (
            f"Patient-initiated messaging (log1p, past {period})"
        ),
        "Encounters, past 7 days": (
            "Recent encounters (log1p, past 7d)"
        ),
    }
    return label_map[term]


table_s6_rows = []

for period in period_order:
    period_results = (
        coefficient_comparison
        .loc[coefficient_comparison["period"].eq(period)]
        .set_index("term")
    )

    for row_number, term in enumerate(term_order):
        if term not in period_results.index:
            raise KeyError(
                f"Missing coefficient-comparison result for "
                f"{period}: {term}"
            )

        result = period_results.loc[term]

        table_s6_rows.append({
            "Message exposure window": (
                period_labels[period] if row_number == 0 else ""
            ),
            "Variable": display_variable_name(term, period),
            "Primary specification (restricted sample) OR (95% CI)": (
                format_or_ci(
                    result["original_OR"],
                    result["original_95CI_low"],
                    result["original_95CI_high"],
                )
            ),
            "Encounter-adjusted specification OR (95% CI)": (
                format_or_ci(
                    result["encounter_adjusted_OR"],
                    result["encounter_adjusted_95CI_low"],
                    result["encounter_adjusted_95CI_high"],
                )
            ),
            "Relative change (%)": format_relative_change(
                result[
                    "OR_percent_change_after_encounter_adjustment"
                ]
            ),
        })


table_s6 = pd.DataFrame(table_s6_rows)

table_s6_title = (
    "Table S6. Comparison of the primary and encounter-adjusted "
    "model specifications."
)

table_s6_note = (
    "Note. Both models were estimated on the same complete-case subset "
    "of HCW-initiated threads to permit a direct comparison between the "
    "primary and encounter-adjusted specifications. The encounter-adjusted "
    "specification additionally included the number of patient encounters "
    "during the 7 days preceding thread initiation, transformed using "
    "log(1+count). Both specifications included patient age, sex, race, "
    "ethnicity, Charlson Comorbidity Index group, prior patient-initiated "
    "messaging, and month-year fixed effects, with standard errors clustered "
    "by patient. Only principal exposure variables are shown. Relative "
    "change was calculated as 100 x (encounter-adjusted OR - primary OR) / "
    "primary OR. HCW indicates health care worker; OR, odds ratio; "
    "CI, confidence interval."
)

display(Markdown(f"**{table_s6_title}**"))
display(table_s6)
display(Markdown(table_s6_note))


# ------------------------------------------------------------
# Save the formatted table and note
# ------------------------------------------------------------

table_s6_path = TABLE_DIR / "Table_S6_primary_vs_encounter_adjusted.csv"
table_s6_note_path = (
    TABLE_DIR / "Table_S6_primary_vs_encounter_adjusted_note.txt"
)

table_s6.to_csv(
    table_s6_path,
    index=False,
    encoding="utf-8-sig",
)

table_s6_note_path.write_text(
    table_s6_title + "\n\n" + table_s6_note,
    encoding="utf-8",
)

print("Saved Table S6:")
print(" -", table_s6_path)
print(" -", table_s6_note_path)

## 5. Primary Encounter-Adjusted Provider/System Models

In [ ]:
import numpy as np
import statsmodels.formula.api as smf


MODELS_BASE  = {}
MODELS_PHYS  = {}
MODELS_OTHER = {}


# =========================
# SAFE MODEL REGISTRY
# =========================
OUTCOME = "patient_replied_7d"
PERIODS = ["30d", "90d"]
AGE_CONT = "Age_at_Message"
RACE_PREFIX = 'C(Race, Treatment(reference="White"))'
ENCOUNTER_CONTROL = "log_encounters_past_7d"

# IMPORTANT: baseline uses Charlson_group without C() in your code
# If Charlson_group is categorical strings, keep as-is.
# If it's numeric and you want categories, change to "C(Charlson_group)".
BASE_CONTROLS_BASELINE = ["Ethnicity", "C(Charlson_group)"]

def joint_wald_p(model, term_substring):
    """Joint Wald p-value for all coeffs whose name contains term_substring."""
    inter_terms = [p for p in model.params.index if term_substring in p]
    if not inter_terms:
        return None
    R = np.zeros((len(inter_terms), len(model.params)))
    idx = list(model.params.index)
    for i, term in enumerate(inter_terms):
        R[i, idx.index(term)] = 1
    return float(model.wald_test(R).pvalue)

def fit_baseline_adjusted_store(
    df_source,
    outcome=OUTCOME,
    periods=PERIODS,
    age_term=AGE_CONT,
    race_prefix=RACE_PREFIX,
    base_controls=BASE_CONTROLS_BASELINE,
    encounter_control=ENCOUNTER_CONTROL,
    store=MODELS_BASE,
    key_prefix="BASE_HCWxSYS",   # <-- Panel A keys
):
    """
    Baseline adjusted model:
      outcome ~ provider + system + provider:system + pinit + encounter control + covariates + month-year FE
    Stores under:
      BASE_HCWxSYS_30d
      BASE_HCWxSYS_90d
    """
    df = df_source.copy()
    df[outcome] = df[outcome].astype(float).fillna(0).astype(int)

    for period in periods:
        x_prov  = f"log_provider_msgs_past_{period}"
        x_sys   = f"log_system_msgs_past_{period}"
        x_pinit = f"log_patient_initiated_threads_past_{period}"

        x_int = f"{x_prov}:{x_sys}"

        # required vars (adjusted)
        na_vars = [
            outcome, "DMRN", "month_year", age_term, "Gender", "Race",
            encounter_control,
            *base_controls,
            x_prov, x_sys, x_pinit
        ]
        na_vars = [v for v in na_vars if v in df.columns]
        df_fit = df.dropna(subset=list(set(na_vars))).copy()

        # adjusted main
        adj_terms = (
            [x_prov, x_sys, x_pinit, x_int, encounter_control] +
            [age_term] +
            ["C(Gender)"] +
            [race_prefix] +
            base_controls +
            ["C(month_year)"]
        )
        adj_formula = f"{outcome} ~ " + " + ".join(adj_terms)

        m = smf.logit(adj_formula, data=df_fit).fit(
            cov_type="cluster",
            cov_kwds={"groups": df_fit["DMRN"]},
            disp=False
        )

        p_int = joint_wald_p(m, x_int)

        key = f"{key_prefix}_{period}"
        store[key] = {
            "key": key,
            "dataset_name": "filtered_df",
            "model": m,
            "period": period,
            "outcome": outcome,
            "N": int(df_fit.shape[0]),
            "p_int": p_int,
            "df_fit": df_fit,
            # keep exact variable names used (for plotting/table)
            "x_prov": x_prov,
            "x_sys": x_sys,
            "x_pinit": x_pinit,
            "x_int": x_int,
            "x_encounter": encounter_control,
        }

        print(f"Stored {key}: N={store[key]['N']:,} | Prov x Sys p={p_int}")

    return store

# ---- RUN (Panel A) ----
fit_baseline_adjusted_store(filtered_df)

from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

def stargazer_baseline_adjusted_from_store(store, key, title, age_term="Age_at_Message"):
    obj = store[key]
    m = obj["model"]
    period = obj["period"]

    x_prov      = obj["x_prov"]
    x_sys       = obj["x_sys"]
    x_int1      = obj["x_int"]
    x_int2      = f"{x_sys}:{x_prov}"   # just in case
    x_pinit     = obj["x_pinit"]
    x_encounter = obj["x_encounter"]

    st = Stargazer([m])
    st.custom_columns([title], [1])
    st.show_confidence_intervals(False)
    st.show_degrees_of_freedom(False)
    st.significant_digits(3)
    st.show_f_statistic = False

    rename = {
        "Intercept": "Constant",
        x_prov:  f"HCW messages (log1p, past {period})",
        x_sys:   f"System messages (log1p, past {period})",
        x_int1:  f"HCW x System (log1p x log1p, past {period})",
        x_int2:  f"HCW x System (log1p x log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",
        x_encounter: "Encounters (log1p, past 7 days)",
        "C(Gender)[T.Male]": "Male",
        age_term: "Age at message (years)",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (Mild)",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (Moderate)",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (Severe)",
    }
    st.rename_covariates(rename)

    # Drop month-year FE rows in display
    idx = [p for p in m.params.index if "C(month_year)" not in p]
    present = set(idx)

    race_prefix_local = 'C(Race, Treatment(reference="White"))'
    eth = sorted([p for p in present if p.startswith("Ethnicity[")])
    cci = sorted([p for p in present if p.startswith("C(Charlson_group)[")])
    race_terms = sorted([p for p in present if p.startswith(race_prefix_local + "[")])

    cov_order = [
        x_prov, x_sys, x_int1, x_int2, x_pinit, x_encounter,
        age_term, "C(Gender)[T.Male]",
    ] + eth + cci + race_terms + ["Intercept"]

    cov_order = [c for c in cov_order if c in present]
    st.covariate_order(cov_order)

    # footer lines (use the stored N/p)
    st.add_line("Month-Year FE (included)", ["Yes"])
    st.add_line("Clustered SE (Patient)", ["Yes"])
    st.add_line("N", [f"{obj['N']:,}"])
    st.add_line("HCW x System p-value", [("" if obj["p_int"] is None else f"{obj['p_int']:.3g}")])

    return st

st30_base = stargazer_baseline_adjusted_from_store(
    MODELS_BASE, "BASE_HCWxSYS_30d",
    title="Baseline (HCW-initiated): Reply within 7 days - 30-day window",
    age_term=AGE_CONT
)

st90_base = stargazer_baseline_adjusted_from_store(
    MODELS_BASE, "BASE_HCWxSYS_90d",
    title="Baseline (HCW-initiated): Reply within 7 days - 90-day window",
    age_term=AGE_CONT
)

html = f"""
<div style='display:flex; justify-content:space-between; gap:18px;'>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>30-day exposure window</div>
    {st30_base.render_html()}
  </div>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>90-day exposure window</div>
    {st90_base.render_html()}
  </div>
</div>
"""
display(HTML(html))

## Time To First Patient Reply ECDF

This supplementary figure uses the exact primary regression sample stored in `MODELS_BASE`.


In [ ]:
# CHUNK 2: TIME-TO-FIRST-PATIENT-REPLY ECDF
# Target notebook: 06_analysis_models_clean.ipynb
# ======================================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


FIGURE_TITLE = (
    "Figure S2. Empirical cumulative distribution of time to first "
    "patient reply following HCW-initiated message threads."
)

FIGURE_CAPTION = (
    "The empirical cumulative distribution function shows the cumulative "
    "proportion of observed first patient replies over time among "
    "HCW-initiated threads in the primary analytic sample. Vertical "
    "reference lines denote the 7-day primary outcome window and the "
    "14-day and 30-day sensitivity-analysis windows. Approximately "
    "87.9%, 94.2%, and 99.8% of observed first replies occurred within "
    "7, 14, and 30 days, respectively. The horizontal axis is displayed "
    "on a log(1+hours) scale to improve visualization of the highly "
    "right-skewed distribution. HCW indicates health care worker."
)

required_model_keys = [
    "BASE_HCWxSYS_30d",
    "BASE_HCWxSYS_90d",
]

missing_model_keys = [
    key for key in required_model_keys
    if key not in MODELS_BASE
]

if missing_model_keys:
    raise KeyError(
        "Run the primary encounter-adjusted model section first. "
        f"Missing model keys: {missing_model_keys}"
    )

primary_30d_sample = (
    MODELS_BASE["BASE_HCWxSYS_30d"]["df_fit"]
    .copy()
)
primary_90d_sample = (
    MODELS_BASE["BASE_HCWxSYS_90d"]["df_fit"]
    .copy()
)

if len(primary_30d_sample) != len(primary_90d_sample):
    raise AssertionError(
        "The primary 30-day and 90-day model samples have "
        "different row counts."
    )

if "thread_id" in primary_30d_sample.columns:
    ids_30d = set(
        primary_30d_sample["thread_id"].astype(str)
    )
    ids_90d = set(
        primary_90d_sample["thread_id"].astype(str)
    )

    if ids_30d != ids_90d:
        raise AssertionError(
            "The primary 30-day and 90-day models do not use "
            "the same thread sample."
        )
elif not primary_30d_sample.index.equals(
    primary_90d_sample.index
):
    raise AssertionError(
        "The primary 30-day and 90-day model samples differ."
    )

reply_sample = primary_30d_sample.copy()

required_reply_columns = {
    "thread_first_time",
    "first_patient_reply_time",
}

missing_reply_columns = (
    required_reply_columns - set(reply_sample.columns)
)

if missing_reply_columns:
    raise KeyError(
        "Primary model sample is missing reply-time columns: "
        + ", ".join(sorted(missing_reply_columns))
    )

reply_sample["thread_first_time"] = pd.to_datetime(
    reply_sample["thread_first_time"],
    errors="coerce",
)

reply_sample["first_patient_reply_time"] = pd.to_datetime(
    reply_sample["first_patient_reply_time"],
    errors="coerce",
)

reply_sample["time_to_first_reply_hours"] = (
    (
        reply_sample["first_patient_reply_time"]
        - reply_sample["thread_first_time"]
    )
    .dt.total_seconds()
    / 3600
)

negative_reply_delays = int(
    reply_sample["time_to_first_reply_hours"]
    .lt(0)
    .fillna(False)
    .sum()
)

if negative_reply_delays:
    raise AssertionError(
        f"{negative_reply_delays:,} first-reply times occurred "
        "before thread initiation."
    )

observed_reply_hours = (
    reply_sample["time_to_first_reply_hours"]
    .dropna()
    .loc[
        lambda values:
            values.ge(0)
    ]
    .astype(float)
)

if observed_reply_hours.empty:
    raise ValueError(
        "No observed patient replies were available for the ECDF."
    )

sorted_reply_hours = np.sort(
    observed_reply_hours.to_numpy()
)

ecdf_y = (
    np.arange(
        1,
        len(sorted_reply_hours) + 1,
    )
    / len(sorted_reply_hours)
    * 100
)

ecdf_x = np.log1p(sorted_reply_hours)

window_thresholds = {
    "7 days (primary)": 7 * 24,
    "14 days": 14 * 24,
    "30 days": 30 * 24,
}

window_styles = {
    "7 days (primary)": {
        "color": "#B2182B",
        "linestyle": "-",
        "linewidth": 1.8,
    },
    "14 days": {
        "color": "#2166AC",
        "linestyle": "--",
        "linewidth": 1.6,
    },
    "30 days": {
        "color": "#555555",
        "linestyle": ":",
        "linewidth": 1.8,
    },
}

threshold_summary_rows = []

for label, threshold_hours in window_thresholds.items():
    n_within = int(
        np.sum(sorted_reply_hours <= threshold_hours)
    )
    pct_within = (
        100 * n_within / len(sorted_reply_hours)
    )

    threshold_summary_rows.append({
        "Reply window": label,
        "Observed first replies within window, n":
            n_within,
        "Observed first replies within window, %":
            pct_within,
    })

reply_window_summary = pd.DataFrame(
    threshold_summary_rows
)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial",
        "Helvetica",
        "DejaVu Sans",
    ],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": ":",
})

fig, ax = plt.subplots(
    figsize=(7.8, 5.2)
)

ax.step(
    ecdf_x,
    ecdf_y,
    where="post",
    color="black",
    linewidth=2.1,
    label="First patient replies",
)

for label, threshold_hours in window_thresholds.items():
    style = window_styles[label]
    threshold_x = np.log1p(threshold_hours)

    ax.axvline(
        threshold_x,
        color=style["color"],
        linestyle=style["linestyle"],
        linewidth=style["linewidth"],
        label=label,
    )

    cumulative_percentage = (
        100
        * np.mean(
            sorted_reply_hours <= threshold_hours
        )
    )

    ax.scatter(
        [threshold_x],
        [cumulative_percentage],
        color=style["color"],
        s=24,
        zorder=4,
    )

tick_hours = np.array([
    0,
    1,
    6,
    24,
    72,
    168,
    336,
    720,
], dtype=float)

tick_labels = [
    "0",
    "1 h",
    "6 h",
    "1 d",
    "3 d",
    "7 d",
    "14 d",
    "30 d",
]

ax.set_xticks(
    np.log1p(tick_hours)
)
ax.set_xticklabels(tick_labels)

ax.set_xlim(
    0,
    np.log1p(30 * 24),
)
ax.set_ylim(0, 100)

ax.set_yticks(
    np.arange(0, 101, 10)
)

ax.set_xlabel(
    "Time to first patient reply (log1p-scaled hours)"
)
ax.set_ylabel(
    "Cumulative percentage of observed first replies"
)

ax.legend(
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.65, 0.01),
)

fig.tight_layout()

FIGURE_OUTPUT_DIR = (
    Path(FIGURE_DIR)
    if "FIGURE_DIR" in globals()
    else Path.cwd()
)
FIGURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_PDF_PATH = (
    FIGURE_OUTPUT_DIR
    / "Figure_S2_time_to_first_patient_reply_ecdf.pdf"
)
FIGURE_PNG_PATH = (
    FIGURE_OUTPUT_DIR
    / "Figure_S2_time_to_first_patient_reply_ecdf.png"
)
FIGURE_SUMMARY_PATH = (
    FIGURE_OUTPUT_DIR
    / "Figure_S2_reply_window_summary.csv"
)

fig.savefig(
    FIGURE_PDF_PATH,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

reply_window_summary.to_csv(
    FIGURE_SUMMARY_PATH,
    index=False,
)

plt.show()

display(reply_window_summary)

display(
    Markdown(
        f"**Caption.** {FIGURE_CAPTION}"
    )
)

print("Saved:", FIGURE_PDF_PATH)
print("Saved:", FIGURE_PNG_PATH)
print("Saved:", FIGURE_SUMMARY_PATH)


In [ ]:
import numpy as np
import pandas as pd

# -------------------------
# JAMA-ish formatting
# -------------------------
def format_p_jama(p):
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "<.001"
    elif p < 0.01:
        return f"{p:.3f}".lstrip("0")
    else:
        return f"{p:.2f}".lstrip("0")


def format_or_ci_jama(or_val, lo, hi, decimals=2):
    if any(pd.isna(x) for x in [or_val, lo, hi]):
        return ""
    return f"{or_val:.{decimals}f} ({lo:.{decimals}f}-{hi:.{decimals}f})"


# ============================================================
# 1) Prettify variable names for stored baseline models
# ============================================================
def prettify_varname_baseline(var, obj):
    period = obj["period"]

    x_prov = obj["x_prov"]
    x_sys = obj["x_sys"]
    x_pinit = obj["x_pinit"]
    x_int1 = obj["x_int"]
    x_int2 = f"{x_sys}:{x_prov}"
    x_encounter = obj.get("x_encounter", "log_encounters_past_7d")

    mapping = {
        # exposure terms
        x_prov: f"HCW messages (log1p, past {period})",
        x_sys: f"System messages (log1p, past {period})",
        x_int1: f"HCW x System (log1p x log1p, past {period})",
        x_int2: f"HCW x System (log1p x log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",

        # recent clinical activity control
        x_encounter: "Encounters (log1p, past 7 days)",

        # adjusted covariates
        "Age_at_Message": "Age at message (years)",
        "C(Gender)[T.Male]": "Male vs Female",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",

        # Charlson group encodings
        "Charlson_group[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "Charlson_group[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "Charlson_group[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
    }

    if var in mapping:
        return mapping[var]

    race_prefix = 'C(Race, Treatment(reference="White"))'
    if var.startswith(race_prefix) and ":" not in var:
        race_cat = var.split("[T.", 1)[1].rstrip("]")
        if race_cat == "Other/Multiple/Unknown":
            race_cat = "Other"
        return f"{race_cat} vs White"

    return var


# ============================================================
# 2) Extract OR table from one stored model
# ============================================================
def extract_or_table_from_store(
    store,
    key,
    model_label,
    drop_intercept=True,
    drop_fe_prefixes=("C(month_year)",),
    decimals=2
):
    obj = store[key]
    model = obj["model"]

    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    rows = []

    for var in params.index:
        if drop_intercept and var == "Intercept":
            continue
        if any(var.startswith(pref) for pref in drop_fe_prefixes):
            continue

        beta = float(params[var])
        lo_beta, hi_beta = conf.loc[var].values.astype(float)

        or_val = float(np.exp(beta))
        lo = float(np.exp(lo_beta))
        hi = float(np.exp(hi_beta))
        p = float(pvals[var])

        rows.append({
            "Model": model_label,
            "Variable": prettify_varname_baseline(var, obj),
            "Odds ratio (95% CI)": format_or_ci_jama(or_val, lo, hi, decimals=decimals),
            "P value": format_p_jama(p),
            "_raw": var
        })

    out = pd.DataFrame(rows)

    # Add N row
    out = pd.concat([
        out,
        pd.DataFrame([{
            "Model": model_label,
            "Variable": "No. of thread anchors (N)",
            "Odds ratio (95% CI)": f"{obj['N']:,}",
            "P value": "",
            "_raw": ""
        }])
    ], ignore_index=True)

    return out[["Model", "Variable", "Odds ratio (95% CI)", "P value", "_raw"]]


# ============================================================
# 3) Build combined OR table for 30d and 90d adjusted models
# ============================================================
table_base_hcw_sys_or = pd.concat([
    extract_or_table_from_store(
        MODELS_BASE,
        "BASE_HCWxSYS_30d",
        model_label="30-day exposure window",
        decimals=2
    ),
    extract_or_table_from_store(
        MODELS_BASE,
        "BASE_HCWxSYS_90d",
        model_label="90-day exposure window",
        decimals=2
    )
], ignore_index=True)


# ============================================================
# 4) Display and QC
# ============================================================
display(table_base_hcw_sys_or.drop(columns=["_raw"]))

unmapped_terms = table_base_hcw_sys_or.query("Variable == _raw")["_raw"].unique()
print("Unmapped terms:", unmapped_terms[:30])

print("Encounter terms in 30d model:", [p for p in MODELS_BASE["BASE_HCWxSYS_30d"]["model"].params.index if "encounter" in p])
print("Encounter terms in 90d model:", [p for p in MODELS_BASE["BASE_HCWxSYS_90d"]["model"].params.index if "encounter" in p])

## 6. Prepare Focused Physician/APP Versus Other-HCW Subsets

In [ ]:
Other_analysis_df = pd.read_parquet(FOCUSED_ANALYSIS_PATH, engine="pyarrow")


Other_analysis_df["Age65"] = pd.Categorical(
    Other_analysis_df["Age65"],
    categories=["<65", "≥65"],
    ordered=False
)

Other_analysis_df["Race"] = pd.Categorical(
    Other_analysis_df["Race"],
    categories=["White", "Black/African American", "Middle Eastern/North African", "Asian", "Native American/Alaskan", "Pacific Islander", "Other/Multiple/Unknown"],
    ordered=False
)

Other_analysis_df["Ethnicity"] = pd.Categorical(
    Other_analysis_df["Ethnicity"],
    categories=["Not Hispanic/Latino", "Hispanic/Latino", "Unknown/Other"],
    ordered=False
)

# Convert pandas StringDtype to a Patsy-compatible categorical dtype.
Other_analysis_df["Gender"] = Other_analysis_df["Gender"].astype("category")

# Ensure Charlson_group is categorical
Other_analysis_df["Charlson_group"] = Other_analysis_df["Charlson_group"].astype("category")

# Ensure CREATED_TIME is datetime
Other_analysis_df["thread_first_time"] = pd.to_datetime(Other_analysis_df["thread_first_time"])
# Create month-year variable for time fixed effects
Other_analysis_df["month_year"] = Other_analysis_df["thread_first_time"].dt.to_period("M").astype(str)


# Collabse Race
import numpy as np

Other_analysis_df["Race"] = np.select(
    [
        Other_analysis_df["Race"].eq("White"),
        Other_analysis_df["Race"].eq("Black/African American"),
        Other_analysis_df["Race"].eq("Asian"),
    ],
    [
        "White",
        "Black/African American",
        "Asian",
    ],
    default="Other/Multiple/Unknown"
)

# (optional) make it a categorical with White as the reference
Other_analysis_df["Race"] = pd.Categorical(
    Other_analysis_df["Race"],
    categories=["White", "Black/African American", "Asian", "Other/Multiple/Unknown"],
    ordered=False
)

# sanity check
tab = Other_analysis_df["Race"].value_counts(dropna=False).to_frame("n")
tab["%"] = (tab["n"] / tab["n"].sum() * 100).round(1)
print(tab)

df = Other_analysis_df.copy()
# Filter out rows with missing Age65
df1 = df.dropna(subset=['Age65']).copy()

print(f"Rows before filtering: {len(df):,}")
print(f"Rows after filtering: {len(df1):,}")
print(f"Rows removed: {len(df) - len(df1):,}")

# Exclude the early and last 2 weeks


# Define date boundaries
start_date = pd.Timestamp("2021-03-01")
end_date = pd.Timestamp("2023-12-31")

# Compute cutoff windows
start_cutoff = start_date + pd.Timedelta(days=30)   # exclude early 2 weeks
end_cutoff = end_date - pd.Timedelta(days=30)       # exclude last 2 weeks

# Filter threads within valid range
filtered_df2 = df1[
    (df1["thread_first_time"] >= start_cutoff) &
    (df1["thread_first_time"] <= end_cutoff)
].copy()

print(f"✅ Kept {len(filtered_df2):,} rows out of {len(df1):,}")
print(f"   Data range kept: {filtered_df2['thread_first_time'].min()} → {filtered_df2['thread_first_time'].max()}")
print("   (Removed threads from the first and last 30 days of data).")

In [ ]:
analysis_physapp = filtered_df2[filtered_df2["thread_initiator_group"] == "PhysAPP"].copy()

In [ ]:
analysis_other = filtered_df2[filtered_df2["thread_initiator_group"] == "OtherHuman"].copy()

## 7. Focused Physician/APP-Initiated Models And OR Table

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# ----------------------------
# Focused model dataset
# ----------------------------
df_foc_phys = analysis_physapp.copy()

OUTCOME = "patient_replied_7d"
PERIODS = ["30d", "90d"]

AGE_CONT = "Age_at_Message"
RACE_PREFIX = 'C(Race, Treatment(reference="White"))'
BASE_CONTROLS = ["Ethnicity", "C(Charlson_group)"]
ENCOUNTER_CONTROL = "log_encounters_past_7d"

# Ensure DV is 0/1 int
df_foc_phys[OUTCOME] = df_foc_phys[OUTCOME].astype(float).fillna(0).astype(int)

def joint_wald_p_contains(model, substring):
    """Joint Wald p-value for all coefficients containing `substring`."""
    terms = [p for p in model.params.index if substring in p]
    if not terms:
        return None
    R = np.zeros((len(terms), len(model.params)))
    idx = list(model.params.index)
    for i, term in enumerate(terms):
        R[i, idx.index(term)] = 1
    return float(model.wald_test(R).pvalue)

def p_for_interaction_two_terms(model, a, b):
    """
    Robust p-value for interaction between a and b,
    whether it appears as 'a:b' or 'b:a'.
    """
    t1 = f"{a}:{b}"
    t2 = f"{b}:{a}"
    if t1 in model.params.index:
        return joint_wald_p_contains(model, t1)
    if t2 in model.params.index:
        return joint_wald_p_contains(model, t2)
    return None

# ----------------------------
# Re-run MAIN models safely
# ----------------------------
for period in PERIODS:
    x_phys  = f"log_physapp_msgs_past_{period}"
    x_othh  = f"log_other_human_msgs_past_{period}"   # other human
    x_sys   = f"log_system_msgs_past_{period}"        # system separate
    x_pinit = f"log_patient_initiated_threads_past_{period}"

    x_int = f"{x_phys}:{x_othh}"

    needed = [
        OUTCOME, "DMRN", "month_year", AGE_CONT, "Gender", "Race", "Ethnicity", "Charlson_group",
        ENCOUNTER_CONTROL,
        x_phys, x_othh, x_sys, x_pinit
    ]
    needed = [c for c in needed if c in df_foc_phys.columns]
    df_fit = df_foc_phys.dropna(subset=needed).copy()

    main_terms = (
        [x_phys, x_othh, x_sys, x_pinit, x_int, ENCOUNTER_CONTROL] +
        [AGE_CONT] +
        ["C(Gender)"] +
        [RACE_PREFIX] +
        BASE_CONTROLS +
        ["C(month_year)"]
    )
    formula = f"{OUTCOME} ~ " + " + ".join(main_terms)

    m = smf.logit(formula, data=df_fit).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_fit["DMRN"]},
        disp=False
    )

    pint = p_for_interaction_two_terms(m, x_phys, x_othh)

    # SAFE names for figure usage
    key = f"FOC_PHYSINIT_{period}"
    MODELS_PHYS[key] = {
        "model": m,
        "df_fit": df_fit,
        "period": period,
        "N": int(df_fit.shape[0]),
        "p_int": pint,
        "x_phys": x_phys,
        "x_other_human": x_othh,
        "x_sys": x_sys,
        "x_pinit": x_pinit,
        "x_encounter": ENCOUNTER_CONTROL,
    }

print("Focused Phys/APP-initiated MAIN models ready for Figure 1B.")
print("30d:", MODELS_PHYS["FOC_PHYSINIT_30d"]["N"], "p_int=", MODELS_PHYS["FOC_PHYSINIT_30d"]["p_int"])
print("90d:", MODELS_PHYS["FOC_PHYSINIT_90d"]["N"], "p_int=", MODELS_PHYS["FOC_PHYSINIT_90d"]["p_int"])

from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

def stargazer_main_from_store(model_store, key, title, age_cont="Age_at_Message"):
    """
    model_store[key] must contain:
      - model, period, N, p_int, x_phys, x_other_human, x_sys, x_pinit
    """
    obj = model_store[key]
    m = obj["model"]
    period = obj["period"]

    x_phys      = obj["x_phys"]
    x_othh      = obj["x_other_human"]
    x_sys       = obj["x_sys"]
    x_pinit     = obj["x_pinit"]
    x_encounter = obj["x_encounter"]

    # interaction could appear either order
    x_int1 = f"{x_phys}:{x_othh}"
    x_int2 = f"{x_othh}:{x_phys}"

    race_prefix_local = 'C(Race, Treatment(reference="White"))'

    st = Stargazer([m])
    st.custom_columns([title], [1])
    st.show_confidence_intervals(False)
    st.show_degrees_of_freedom(False)
    st.significant_digits(3)
    st.show_f_statistic = False

    rename = {
        "Intercept": "Constant",
        x_phys:  f"Physician/APP messages (log1p, past {period})",
        x_othh:  f"Other human messages (log1p, past {period})",
        x_sys:   f"System messages (log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",
        x_int1:  f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_int2:  f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_encounter: "Encounters (log1p, past 7 days)",
        "C(Gender)[T.Male]": "Male vs Female",
        age_cont: "Age at message (years)",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (Mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (Moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (Severe) vs 0",
    }
    st.rename_covariates(rename)

    # remove FE from display + build covariate order
    idx = [p for p in m.params.index if "C(month_year)" not in p]
    present = set(idx)

    race_terms = sorted([p for p in idx if p.startswith(race_prefix_local + "[")])
    eth_terms  = sorted([p for p in idx if p.startswith("Ethnicity[")])
    cci_terms  = sorted([p for p in idx if p.startswith("C(Charlson_group)[") and ":" not in p])

    cov_order = [
        x_phys,
        x_othh,
        x_int1, x_int2,
        x_sys,
        x_pinit,
        x_encounter,
        age_cont,
        "C(Gender)[T.Male]",
    ] + eth_terms + cci_terms + race_terms + ["Intercept"]

    cov_order = [c for c in cov_order if c in present]
    st.covariate_order(cov_order)

    # footer
    st.add_line("Sample restriction", ["Phys/APP-initiated threads"])
    st.add_line("Month-Year FE (included)", ["Yes"])
    st.add_line("Clustered SE (Patient)", ["Yes"])
    st.add_line("N", [f"{obj.get('N', ''):,}" if obj.get("N") is not None else ""])
    st.add_line("Phys/APP x Other human p-value", [("" if obj.get("p_int") is None else f"{obj['p_int']:.3g}")])

    return st


# ---- Build and display 30d + 90d side-by-side from your safe store ----
st30 = stargazer_main_from_store(
    MODELS_PHYS, "FOC_PHYSINIT_30d",
    title="Focused (Phys/APP-initiated): Reply within 7 days - 30-day window",
    age_cont=AGE_CONT
)

st90 = stargazer_main_from_store(
    MODELS_PHYS, "FOC_PHYSINIT_90d",
    title="Focused (Phys/APP-initiated): Reply within 7 days - 90-day window",
    age_cont=AGE_CONT
)

html = f"""
<div style='display:flex; justify-content:space-between; gap:18px;'>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>30-day exposure window</div>
    {st30.render_html()}
  </div>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>90-day exposure window</div>
    {st90.render_html()}
  </div>
</div>
"""
display(HTML(html))

In [ ]:
import numpy as np
import pandas as pd

# -------------------------
# JAMA-ish formatting
# -------------------------
def format_p_jama(p):
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "<.001"
    elif p < 0.01:
        return f"{p:.3f}".lstrip("0")
    else:
        return f"{p:.2f}".lstrip("0")


def format_or_ci_jama(or_val, lo, hi, decimals=2):
    if any(pd.isna(x) for x in [or_val, lo, hi]):
        return ""
    return f"{or_val:.{decimals}f} ({lo:.{decimals}f}-{hi:.{decimals}f})"


# ============================================================
# 1) Prettify variable names for stored Phys/APP-focused models
# ============================================================
def prettify_varname_physapp(var, obj):
    period = obj["period"]

    x_phys = obj["x_phys"]
    x_othh = obj["x_other_human"]
    x_sys = obj["x_sys"]
    x_pinit = obj["x_pinit"]
    x_int1 = f"{x_phys}:{x_othh}"
    x_int2 = f"{x_othh}:{x_phys}"
    x_encounter = obj.get("x_encounter", "log_encounters_past_7d")

    mapping = {
        # exposure terms
        x_phys: f"Physician/APP messages (log1p, past {period})",
        x_othh: f"Other human messages (log1p, past {period})",
        x_sys: f"System messages (log1p, past {period})",
        x_int1: f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_int2: f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",

        # recent clinical activity control
        x_encounter: "Encounters (log1p, past 7 days)",

        # adjusted covariates
        "Age_at_Message": "Age at message (years)",
        "C(Gender)[T.Male]": "Male vs Female",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",

        # Charlson group encodings
        "Charlson_group[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "Charlson_group[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "Charlson_group[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
    }

    if var in mapping:
        return mapping[var]

    race_prefix = 'C(Race, Treatment(reference="White"))'
    if var.startswith(race_prefix) and ":" not in var:
        race_cat = var.split("[T.", 1)[1].rstrip("]")
        if race_cat == "Other/Multiple/Unknown":
            race_cat = "Other"
        return f"{race_cat} vs White"

    return var


# ============================================================
# 2) Extract OR table from one stored model
# ============================================================
def extract_or_table_from_physapp_store(
    store,
    key,
    model_label,
    drop_intercept=True,
    drop_fe_prefixes=("C(month_year)",),
    decimals=2
):
    obj = store[key]
    model = obj["model"]

    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    rows = []

    for var in params.index:
        if drop_intercept and var == "Intercept":
            continue
        if any(var.startswith(pref) for pref in drop_fe_prefixes):
            continue

        beta = float(params[var])
        lo_beta, hi_beta = conf.loc[var].values.astype(float)

        or_val = float(np.exp(beta))
        lo = float(np.exp(lo_beta))
        hi = float(np.exp(hi_beta))
        p = float(pvals[var])

        rows.append({
            "Model": model_label,
            "Variable": prettify_varname_physapp(var, obj),
            "Odds ratio (95% CI)": format_or_ci_jama(or_val, lo, hi, decimals=decimals),
            "P value": format_p_jama(p),
            "_raw": var
        })

    out = pd.DataFrame(rows)

    # Add N row
    out = pd.concat([
        out,
        pd.DataFrame([{
            "Model": model_label,
            "Variable": "No. of thread anchors (N)",
            "Odds ratio (95% CI)": f"{obj['N']:,}",
            "P value": "",
            "_raw": ""
        }])
    ], ignore_index=True)

    return out[["Model", "Variable", "Odds ratio (95% CI)", "P value", "_raw"]]


# ============================================================
# 3) Build combined OR table for 30d and 90d Phys/APP models
# ============================================================
table_physapp_or = pd.concat([
    extract_or_table_from_physapp_store(
        MODELS_PHYS,
        "FOC_PHYSINIT_30d",
        model_label="30-day exposure window",
        decimals=2
    ),
    extract_or_table_from_physapp_store(
        MODELS_PHYS,
        "FOC_PHYSINIT_90d",
        model_label="90-day exposure window",
        decimals=2
    )
], ignore_index=True)


# ============================================================
# 4) Display and QC
# ============================================================
display(table_physapp_or.drop(columns=["_raw"]))

unmapped_terms = table_physapp_or.query("Variable == _raw")["_raw"].unique()
print("Unmapped terms:", unmapped_terms[:30])

print("Encounter terms in 30d model:", [p for p in MODELS_PHYS["FOC_PHYSINIT_30d"]["model"].params.index if "encounter" in p])
print("Encounter terms in 90d model:", [p for p in MODELS_PHYS["FOC_PHYSINIT_90d"]["model"].params.index if "encounter" in p])

## 8. Focused Other-HCW-Initiated Models And OR Table

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# =========================
# SAFE MODEL REGISTRY
# =========================

OUTCOME = "patient_replied_7d"
PERIODS = ["30d", "90d"]
AGE_CONT = "Age_at_Message"
RACE_PREFIX = 'C(Race, Treatment(reference="White"))'
BASE_CONTROLS = ["Ethnicity", "C(Charlson_group)"]
ENCOUNTER_CONTROL = "log_encounters_past_7d"

def joint_wald_p(model, term_substring):
    """Joint Wald p-value for all coeffs whose name contains term_substring."""
    inter_terms = [p for p in model.params.index if term_substring in p]
    if not inter_terms:
        return None
    R = np.zeros((len(inter_terms), len(model.params)))
    idx = list(model.params.index)
    for i, term in enumerate(inter_terms):
        R[i, idx.index(term)] = 1
    return float(model.wald_test(R).pvalue)

def fit_main_other_nonphys_models_store(
    df_source,
    outcome=OUTCOME,
    periods=PERIODS,
    age_cont=AGE_CONT,
    race_prefix=RACE_PREFIX,
    base_controls=BASE_CONTROLS,
    encounter_control=ENCOUNTER_CONTROL,
    store=MODELS_OTHER,
    key_prefix="FOC_NONPHYSINIT",   # <-- unique prefix for analysis_other
):
    """
    Fits MAIN models predicting reply to non-Phys/APP-initiated threads
    using separate 'other human' + 'system' exposures and their interaction with Phys/APP.
    Stores models under:
      FOC_NONPHYSINIT_30d
      FOC_NONPHYSINIT_90d
    """
    df = df_source.copy()
    df[outcome] = df[outcome].astype(float).fillna(0).astype(int)

    for period in periods:
        # exposures (must exist in analysis_other)
        x_phys  = f"log_physapp_msgs_past_{period}"
        x_othh  = f"log_other_human_msgs_past_{period}"
        x_sys   = f"log_system_msgs_past_{period}"
        x_pinit = f"log_patient_initiated_threads_past_{period}"

        x_int = f"{x_phys}:{x_othh}"

        # required vars
        na_vars = [
            outcome, "DMRN", "month_year", age_cont, "Gender", "Race", "Ethnicity", "Charlson_group",
            encounter_control,
            x_phys, x_othh, x_sys, x_pinit
        ]
        na_vars = [v for v in na_vars if v in df.columns]
        df_fit = df.dropna(subset=list(set(na_vars))).copy()

        # MAIN model
        main_terms = (
            [x_phys, x_othh, x_sys, x_pinit, x_int, encounter_control] +
            [age_cont] +
            ["C(Gender)"] +
            [race_prefix] +
            base_controls +
            ["C(month_year)"]
        )
        main_formula = f"{outcome} ~ " + " + ".join(main_terms)

        m = smf.logit(main_formula, data=df_fit).fit(
            cov_type="cluster",
            cov_kwds={"groups": df_fit["DMRN"]},
            disp=False
        )

        p_int = joint_wald_p(m, x_int)

        key = f"{key_prefix}_{period}"
        store[key] = {
            "key": key,
            "dataset_name": "analysis_other",
            "model": m,
            "period": period,
            "outcome": outcome,
            "N": int(df_fit.shape[0]),
            "p_int": p_int,

            # store variable names used (plotting/table code can read these)
            "x_phys": x_phys,
            "x_other_human": x_othh,
            "x_sys": x_sys,
            "x_pinit": x_pinit,
            "x_int": x_int,
            "x_encounter": encounter_control,
        }

        print(f"Stored {key}: N={store[key]['N']:,} | Phys x OtherHuman p={p_int}")

    return store

# ---- RUN for your non-Phys/APP initiated dataset ----
fit_main_other_nonphys_models_store(analysis_other, store=MODELS_OTHER)

from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

def stargazer_main_from_store(model_store, key, title, age_cont="Age_at_Message"):
    """
    model_store[key] must contain:
      - model, period, N, p_int, x_phys, x_other_human, x_sys, x_pinit
    """
    obj = model_store[key]
    m = obj["model"]
    period = obj["period"]

    x_phys      = obj["x_phys"]
    x_othh      = obj["x_other_human"]
    x_sys       = obj["x_sys"]
    x_pinit     = obj["x_pinit"]
    x_encounter = obj["x_encounter"]

    # interaction could appear either order
    x_int1 = f"{x_phys}:{x_othh}"
    x_int2 = f"{x_othh}:{x_phys}"

    race_prefix_local = 'C(Race, Treatment(reference="White"))'

    st = Stargazer([m])
    st.custom_columns([title], [1])
    st.show_confidence_intervals(False)
    st.show_degrees_of_freedom(False)
    st.significant_digits(3)
    st.show_f_statistic = False

    rename = {
        "Intercept": "Constant",
        x_phys:  f"Physician/APP messages (log1p, past {period})",
        x_othh:  f"Other human messages (log1p, past {period})",
        x_sys:   f"System messages (log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",
        x_int1:  f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_int2:  f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_encounter: "Encounters (log1p, past 7 days)",
        "C(Gender)[T.Male]": "Male vs Female",
        age_cont: "Age at message (years)",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (Mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (Moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (Severe) vs 0",
    }
    st.rename_covariates(rename)

    # remove FE from display + build covariate order
    idx = [p for p in m.params.index if "C(month_year)" not in p]
    present = set(idx)

    race_terms = sorted([p for p in idx if p.startswith(race_prefix_local + "[")])
    eth_terms  = sorted([p for p in idx if p.startswith("Ethnicity[")])
    cci_terms  = sorted([p for p in idx if p.startswith("C(Charlson_group)[") and ":" not in p])

    cov_order = [
        x_phys,
        x_othh,
        x_int1, x_int2,
        x_sys,
        x_pinit,
        x_encounter,
        age_cont,
        "C(Gender)[T.Male]",
    ] + eth_terms + cci_terms + race_terms + ["Intercept"]

    cov_order = [c for c in cov_order if c in present]
    st.covariate_order(cov_order)

    # footer
    st.add_line("Sample restriction", ["Other HCW-initiated threads"])
    st.add_line("Month-Year FE (included)", ["Yes"])
    st.add_line("Clustered SE (Patient)", ["Yes"])
    st.add_line("N", [f"{obj.get('N', ''):,}" if obj.get("N") is not None else ""])
    st.add_line("Phys/APP x Other human p-value", [("" if obj.get("p_int") is None else f"{obj['p_int']:.3g}")])

    return st


# ---- Build and display 30d + 90d side-by-side from your safe store ----

st30_nonphys = stargazer_main_from_store(
    MODELS_OTHER, "FOC_NONPHYSINIT_30d",
    title="Focused (non-Phys/APP-initiated): Reply within 7 days - 30-day window",
    age_cont=AGE_CONT
)

st90_nonphys = stargazer_main_from_store(
    MODELS_OTHER, "FOC_NONPHYSINIT_90d",
    title="Focused (non-Phys/APP-initiated): Reply within 7 days - 90-day window",
    age_cont=AGE_CONT
)

html = f"""
<div style='display:flex; justify-content:space-between; gap:18px;'>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>30-day exposure window</div>
    {st30_nonphys.render_html()}
  </div>
  <div style='width:49%;'>
    <div style='font-weight:600; font-size:16px; margin-bottom:6px;'>90-day exposure window</div>
    {st90_nonphys.render_html()}
  </div>
</div>
"""
display(HTML(html))

In [ ]:
import numpy as np
import pandas as pd

# -------------------------
# JAMA-ish formatting
# -------------------------
def format_p_jama(p):
    if p is None or pd.isna(p):
        return ""
    p = float(p)
    if p < 0.001:
        return "<.001"
    elif p < 0.01:
        return f"{p:.3f}".lstrip("0")
    else:
        return f"{p:.2f}".lstrip("0")


def format_or_ci_jama(or_val, lo, hi, decimals=2):
    if any(pd.isna(x) for x in [or_val, lo, hi]):
        return ""
    return f"{or_val:.{decimals}f} ({lo:.{decimals}f}-{hi:.{decimals}f})"


# ============================================================
# 1) Prettify variable names for stored non-Phys/APP models
# ============================================================
def prettify_varname_nonphys(var, obj):
    period = obj["period"]

    x_phys = obj["x_phys"]
    x_othh = obj["x_other_human"]
    x_sys = obj["x_sys"]
    x_pinit = obj["x_pinit"]
    x_int1 = f"{x_phys}:{x_othh}"
    x_int2 = f"{x_othh}:{x_phys}"
    x_encounter = obj.get("x_encounter", "log_encounters_past_7d")

    mapping = {
        # exposure terms
        x_phys: f"Physician/APP messages (log1p, past {period})",
        x_othh: f"Other human messages (log1p, past {period})",
        x_sys: f"System messages (log1p, past {period})",
        x_int1: f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_int2: f"Physician/APP x Other human (log1p x log1p, past {period})",
        x_pinit: f"Patient-initiated threads (log1p, past {period})",

        # recent clinical activity control
        x_encounter: "Encounters (log1p, past 7 days)",

        # adjusted covariates
        "Age_at_Message": "Age at message (years)",
        "C(Gender)[T.Male]": "Male vs Female",
        "Ethnicity[T.Hispanic/Latino]": "Hispanic/Latino vs non-Hispanic",
        "Ethnicity[T.Unknown/Other]": "Unknown/Other vs non-Hispanic",

        # Charlson group encodings
        "Charlson_group[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "Charlson_group[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "Charlson_group[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
        "C(Charlson_group)[T.1-2 (Mild)]": "CCI: 1-2 (mild) vs 0",
        "C(Charlson_group)[T.3-4 (Moderate)]": "CCI: 3-4 (moderate) vs 0",
        "C(Charlson_group)[T.5+ (Severe)]": "CCI: 5+ (severe) vs 0",
    }

    if var in mapping:
        return mapping[var]

    race_prefix = 'C(Race, Treatment(reference="White"))'
    if var.startswith(race_prefix) and ":" not in var:
        race_cat = var.split("[T.", 1)[1].rstrip("]")
        if race_cat == "Other/Multiple/Unknown":
            race_cat = "Other"
        return f"{race_cat} vs White"

    return var


# ============================================================
# 2) Extract OR table from one stored model
# ============================================================
def extract_or_table_from_nonphys_store(
    store,
    key,
    model_label,
    drop_intercept=True,
    drop_fe_prefixes=("C(month_year)",),
    decimals=2
):
    obj = store[key]
    model = obj["model"]

    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    rows = []

    for var in params.index:
        if drop_intercept and var == "Intercept":
            continue
        if any(var.startswith(pref) for pref in drop_fe_prefixes):
            continue

        beta = float(params[var])
        lo_beta, hi_beta = conf.loc[var].values.astype(float)

        or_val = float(np.exp(beta))
        lo = float(np.exp(lo_beta))
        hi = float(np.exp(hi_beta))
        p = float(pvals[var])

        rows.append({
            "Model": model_label,
            "Variable": prettify_varname_nonphys(var, obj),
            "Odds ratio (95% CI)": format_or_ci_jama(or_val, lo, hi, decimals=decimals),
            "P value": format_p_jama(p),
            "_raw": var
        })

    out = pd.DataFrame(rows)

    # Add N row
    out = pd.concat([
        out,
        pd.DataFrame([{
            "Model": model_label,
            "Variable": "No. of thread anchors (N)",
            "Odds ratio (95% CI)": f"{obj['N']:,}",
            "P value": "",
            "_raw": ""
        }])
    ], ignore_index=True)

    return out[["Model", "Variable", "Odds ratio (95% CI)", "P value", "_raw"]]


# ============================================================
# 3) Build combined OR table for 30d and 90d non-Phys/APP models
# ============================================================
table_nonphys_or = pd.concat([
    extract_or_table_from_nonphys_store(
        MODELS_OTHER,
        "FOC_NONPHYSINIT_30d",
        model_label="30-day exposure window",
        decimals=2
    ),
    extract_or_table_from_nonphys_store(
        MODELS_OTHER,
        "FOC_NONPHYSINIT_90d",
        model_label="90-day exposure window",
        decimals=2
    )
], ignore_index=True)


# ============================================================
# 4) Display and QC
# ============================================================
display(table_nonphys_or.drop(columns=["_raw"]))

unmapped_terms = table_nonphys_or.query("Variable == _raw")["_raw"].unique()
print("Unmapped terms:", unmapped_terms[:30])

print("Encounter terms in 30d model:", [p for p in MODELS_OTHER["FOC_NONPHYSINIT_30d"]["model"].params.index if "encounter" in p])
print("Encounter terms in 90d model:", [p for p in MODELS_OTHER["FOC_NONPHYSINIT_90d"]["model"].params.index if "encounter" in p])

## Table 3

In [ ]:
# ============================================================
# Table 3. Primary encounter-adjusted models
# ============================================================

from html import escape
from IPython.display import HTML, display
import numpy as np
import pandas as pd


# Use the stored encounter-adjusted 30-day models.
table3_objects = [
    MODELS_BASE["BASE_HCWxSYS_30d"],
    MODELS_PHYS["FOC_PHYSINIT_30d"],
    MODELS_OTHER["FOC_NONPHYSINIT_30d"],
]

table3_models = [obj["model"] for obj in table3_objects]
table3_sample_sizes = [int(obj["N"]) for obj in table3_objects]


def find_interaction_term(model, first_term, second_term):
    """Return the interaction term regardless of its ordering."""
    candidates = [
        f"{first_term}:{second_term}",
        f"{second_term}:{first_term}",
    ]

    for candidate in candidates:
        if candidate in model.params.index:
            return candidate

    raise KeyError(
        f"Interaction term not found: {first_term} x {second_term}"
    )


def format_model_result(model, term):
    """Return formatted OR (95% CI) and P value."""
    if term is None:
        return "NA", "NA"

    if term not in model.params.index:
        raise KeyError(f"Term not found in model: {term}")

    beta = float(model.params[term])
    interval = model.conf_int().loc[term].astype(float)

    odds_ratio = float(np.exp(beta))
    lower = float(np.exp(interval.iloc[0]))
    upper = float(np.exp(interval.iloc[1]))
    p_value = float(model.pvalues[term])

    or_ci = f"{odds_ratio:.2f} ({lower:.2f}–{upper:.2f})"

    if p_value < 0.001:
        p_text = "<.001"
    elif p_value < 0.01:
        p_text = f"{p_value:.3f}".lstrip("0")
    else:
        p_text = f"{p_value:.2f}".lstrip("0")

    return or_ci, p_text


base_obj, phys_obj, nonphys_obj = table3_objects
base_model, phys_model, nonphys_model = table3_models

race_prefix = 'C(Race, Treatment(reference="White"))'

base_interaction = find_interaction_term(
    base_model,
    base_obj["x_prov"],
    base_obj["x_sys"],
)

phys_interaction = find_interaction_term(
    phys_model,
    phys_obj["x_phys"],
    phys_obj["x_other_human"],
)

nonphys_interaction = find_interaction_term(
    nonphys_model,
    nonphys_obj["x_phys"],
    nonphys_obj["x_other_human"],
)


# ------------------------------------------------------------
# Define rows in manuscript order
# ------------------------------------------------------------

table3_row_definitions = [
    {
        "label": "Age at message (year)",
        "terms": ["Age_at_Message"] * 3,
    },
    {
        "label": "Sex (reference: female)",
        "section": True,
    },
    {
        "label": "Male",
        "indent": True,
        "terms": ["C(Gender)[T.Male]"] * 3,
    },
    {
        "label": "Race (reference: White)",
        "section": True,
    },
    {
        "label": "Asian",
        "indent": True,
        "terms": [f"{race_prefix}[T.Asian]"] * 3,
    },
    {
        "label": "Black or African American",
        "indent": True,
        "terms": [
            f"{race_prefix}[T.Black/African American]"
        ] * 3,
    },
    {
        "label": "Other",
        "indent": True,
        "terms": [
            f"{race_prefix}[T.Other/Multiple/Unknown]"
        ] * 3,
    },
    {
        "label": "Ethnicity (reference: non-Hispanic)",
        "section": True,
    },
    {
        "label": "Hispanic/Latino",
        "indent": True,
        "terms": ["Ethnicity[T.Hispanic/Latino]"] * 3,
    },
    {
        "label": "Unknown",
        "indent": True,
        "terms": ["Ethnicity[T.Unknown/Other]"] * 3,
    },
    {
        "label": (
            "Charlson Comorbidity Index "
            "(reference: CCI 0)"
        ),
        "section": True,
    },
    {
        "label": "1–2 (mild)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.1-2 (Mild)]"
        ] * 3,
    },
    {
        "label": "3–4 (moderate)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.3-4 (Moderate)]"
        ] * 3,
    },
    {
        "label": "≥5 (severe)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.5+ (Severe)]"
        ] * 3,
    },
    {
        "label": (
            "Patient-initiated messaging "
            "(log, past 30 days)"
        ),
        "terms": [
            obj["x_pinit"] for obj in table3_objects
        ],
    },
    {
        "label": (
            "Recent healthcare encounters "
            "(log1p, past 7 days)"
        ),
        "terms": [
            obj["x_encounter"] for obj in table3_objects
        ],
    },
    {
        "label": "Message exposures (log, past 30 days)",
        "section": True,
    },
    {
        "label": "System",
        "indent": True,
        "terms": [
            base_obj["x_sys"],
            phys_obj["x_sys"],
            nonphys_obj["x_sys"],
        ],
    },
    {
        "label": "HCW",
        "indent": True,
        "terms": [
            base_obj["x_prov"],
            None,
            None,
        ],
    },
    {
        "label": "HCW × system",
        "indent": True,
        "terms": [
            base_interaction,
            None,
            None,
        ],
    },
    {
        "label": "Physician/APP",
        "indent": True,
        "terms": [
            None,
            phys_obj["x_phys"],
            nonphys_obj["x_phys"],
        ],
    },
    {
        "label": "Other HCW",
        "indent": True,
        "terms": [
            None,
            phys_obj["x_other_human"],
            nonphys_obj["x_other_human"],
        ],
    },
    {
        "label": "Physician/APP × Other HCW",
        "indent": True,
        "terms": [
            None,
            phys_interaction,
            nonphys_interaction,
        ],
    },
]


# ------------------------------------------------------------
# Extract model estimates
# ------------------------------------------------------------

table3_records = []

for row in table3_row_definitions:
    record = {
        "Variable": row["label"],
        "section": bool(row.get("section", False)),
        "indent": bool(row.get("indent", False)),
    }

    if record["section"]:
        estimates = [("", "")] * 3
    else:
        estimates = [
            format_model_result(model, term)
            for model, term in zip(
                table3_models,
                row["terms"],
            )
        ]

    for model_number, (or_ci, p_value) in enumerate(
        estimates,
        start=1,
    ):
        record[
            f"Model {model_number} OR (95% CI)"
        ] = or_ci
        record[
            f"Model {model_number} P value"
        ] = p_value

    table3_records.append(record)


table3_display = pd.DataFrame(table3_records)

table3_export = table3_display.drop(
    columns=["section", "indent"]
).copy()


# ------------------------------------------------------------
# Table title and note
# ------------------------------------------------------------

table3_title = (
    "Table 3. Multivariable logistic regression of patient "
    "reply within 7 days to HCW-initiated threads "
    "(30-day exposure)"
)

table3_note = (
    "Model covariates included patient age, sex, race, "
    "ethnicity, Charlson Comorbidity Index (CCI), prior "
    "patient-initiated messaging in the preceding 30 days, "
    "and recent healthcare encounters during the 7 days "
    "preceding thread initiation (log[1+count]). Models "
    "incorporated month-year fixed effects to account for "
    "temporal trends with robust standard errors clustered "
    "at the patient level. Reference groups were female "
    "(sex), White (race), non-Hispanic (ethnicity), and CCI "
    "0 (none) for the comorbidity groups. Other race includes "
    "unknown or multiple races, as well as small-cell "
    "categories. Message exposure variables and "
    "patient-initiated messaging were transformed using the "
    "natural logarithm log(1+count). Other HCW indicates "
    "non-physician/APP health care workers. NA indicates a "
    "variable not included in the model. All tests were "
    "2-tailed, with statistical significance defined as "
    "P<.05. Abbreviations: OR, odds ratio; CI, confidence "
    "interval; HCW, health care worker; APP, advanced "
    "practice provider."
)


model_headers = [
    (
        "Reply to any HCW-initiated threads"
        f"<br>(n = {table3_sample_sizes[0]:,})"
    ),
    (
        "Reply to physician/APP-initiated threads"
        f"<br>(n = {table3_sample_sizes[1]:,})"
    ),
    (
        "Reply to non-physician/APP HCW-initiated threads"
        f"<br>(n = {table3_sample_sizes[2]:,})"
    ),
]


# ------------------------------------------------------------
# Build publication-formatted HTML
# ------------------------------------------------------------

body_rows = []

for _, row in table3_display.iterrows():
    label = escape(str(row["Variable"]))

    if row["section"]:
        body_rows.append(
            '<tr class="section-row">'
            f"<td>{label}</td>"
            '<td colspan="6"></td>'
            "</tr>"
        )
        continue

    label_class = "indent" if row["indent"] else ""
    cells = [f'<td class="{label_class}">{label}</td>']

    for model_number in range(1, 4):
        or_ci = escape(
            str(
                row[
                    f"Model {model_number} OR (95% CI)"
                ]
            )
        )
        p_value = escape(
            str(
                row[
                    f"Model {model_number} P value"
                ]
            )
        )

        cells.append(
            f'<td class="number">{or_ci}</td>'
        )
        cells.append(
            f'<td class="number">{p_value}</td>'
        )

    body_rows.append(
        "<tr>" + "".join(cells) + "</tr>"
    )


table3_html = f"""
<style>
.table3-wrap {{
    font-family: "Times New Roman", serif;
    max-width: 1100px;
    color: #000;
}}
.table3-title {{
    font-size: 10.5pt;
    font-weight: bold;
    margin-bottom: 6px;
}}
.table3 {{
    border-collapse: collapse;
    width: 100%;
    font-size: 9.5pt;
    line-height: 1.05;
}}
.table3 th {{
    font-weight: bold;
    text-align: center;
    padding: 3px 4px;
    vertical-align: bottom;
}}
.table3 thead tr:first-child th {{
    border-top: 1.2px solid #000;
}}
.table3 thead tr:last-child th {{
    border-bottom: 1px solid #000;
}}
.table3 td {{
    padding: 2px 4px;
    vertical-align: top;
}}
.table3 tbody tr:last-child td {{
    border-bottom: 1.2px solid #000;
}}
.table3 .number {{
    text-align: center;
    white-space: nowrap;
}}
.table3 .indent {{
    padding-left: 18px;
}}
.table3 .section-row td:first-child {{
    font-weight: bold;
    padding-top: 5px;
}}
.table3-note {{
    font-size: 9pt;
    line-height: 1.15;
    margin-top: 8px;
    text-align: left;
}}
</style>

<div class="table3-wrap">
  <div class="table3-title">
    {escape(table3_title)}
  </div>

  <table class="table3">
    <thead>
      <tr>
        <th rowspan="2" style="text-align:left">
          Variable
        </th>
        <th colspan="2">{model_headers[0]}</th>
        <th colspan="2">{model_headers[1]}</th>
        <th colspan="2">{model_headers[2]}</th>
      </tr>
      <tr>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
      </tr>
    </thead>

    <tbody>
      {''.join(body_rows)}
    </tbody>
  </table>

  <div class="table3-note">
    {escape(table3_note)}
  </div>
</div>
"""


display(HTML(table3_html))


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

table3_csv_path = (
    TABLE_DIR
    / "Table_3_primary_encounter_adjusted_30d.csv"
)

table3_html_path = (
    TABLE_DIR
    / "Table_3_primary_encounter_adjusted_30d.html"
)

table3_note_path = (
    TABLE_DIR
    / "Table_3_primary_encounter_adjusted_30d_note.txt"
)

table3_export.to_csv(
    table3_csv_path,
    index=False,
    encoding="utf-8-sig",
)

table3_html_path.write_text(
    table3_html,
    encoding="utf-8",
)

table3_note_path.write_text(
    table3_title + "\n\n" + table3_note,
    encoding="utf-8",
)

print("Saved Table 3:")
for path in [
    table3_csv_path,
    table3_html_path,
    table3_note_path,
]:
    print(" -", path)

In [ ]:
# Table S1. Encounter-adjusted 90-day sensitivity analysis
# ============================================================

from html import escape
from IPython.display import HTML, display
import numpy as np
import pandas as pd


table_s1_objects = [
    MODELS_BASE["BASE_HCWxSYS_90d"],
    MODELS_PHYS["FOC_PHYSINIT_90d"],
    MODELS_OTHER["FOC_NONPHYSINIT_90d"],
]

table_s1_models = [obj["model"] for obj in table_s1_objects]
table_s1_sample_sizes = [
    int(obj["N"]) for obj in table_s1_objects
]


def s1_find_interaction(model, first_term, second_term):
    candidates = [
        f"{first_term}:{second_term}",
        f"{second_term}:{first_term}",
    ]

    for candidate in candidates:
        if candidate in model.params.index:
            return candidate

    raise KeyError(
        f"Interaction not found: {first_term} x {second_term}"
    )


def s1_format_result(model, term):
    if term is None:
        return "NA", "NA"

    if term not in model.params.index:
        raise KeyError(f"Term not found in model: {term}")

    beta = float(model.params[term])
    interval = model.conf_int().loc[term].astype(float)

    odds_ratio = float(np.exp(beta))
    lower = float(np.exp(interval.iloc[0]))
    upper = float(np.exp(interval.iloc[1]))
    p_value = float(model.pvalues[term])

    or_ci = f"{odds_ratio:.2f} ({lower:.2f}–{upper:.2f})"

    if p_value < 0.001:
        p_text = "<.001"
    elif p_value < 0.01:
        p_text = f"{p_value:.3f}".lstrip("0")
    else:
        p_text = f"{p_value:.2f}".lstrip("0")

    return or_ci, p_text


base_obj, phys_obj, nonphys_obj = table_s1_objects
base_model, phys_model, nonphys_model = table_s1_models

race_prefix = 'C(Race, Treatment(reference="White"))'

base_interaction = s1_find_interaction(
    base_model,
    base_obj["x_prov"],
    base_obj["x_sys"],
)

phys_interaction = s1_find_interaction(
    phys_model,
    phys_obj["x_phys"],
    phys_obj["x_other_human"],
)

nonphys_interaction = s1_find_interaction(
    nonphys_model,
    nonphys_obj["x_phys"],
    nonphys_obj["x_other_human"],
)


# ------------------------------------------------------------
# Define rows in the original Table S1 order
# ------------------------------------------------------------

table_s1_row_definitions = [
    {
        "label": "Age at message (year)",
        "terms": ["Age_at_Message"] * 3,
    },
    {
        "label": "Sex (reference: female)",
        "section": True,
    },
    {
        "label": "Male",
        "indent": True,
        "terms": ["C(Gender)[T.Male]"] * 3,
    },
    {
        "label": "Race (reference: White)",
        "section": True,
    },
    {
        "label": "Asian",
        "indent": True,
        "terms": [f"{race_prefix}[T.Asian]"] * 3,
    },
    {
        "label": "Black or African American",
        "indent": True,
        "terms": [
            f"{race_prefix}[T.Black/African American]"
        ] * 3,
    },
    {
        "label": "Other",
        "indent": True,
        "terms": [
            f"{race_prefix}[T.Other/Multiple/Unknown]"
        ] * 3,
    },
    {
        "label": "Ethnicity (reference: non-Hispanic)",
        "section": True,
    },
    {
        "label": "Hispanic/Latino",
        "indent": True,
        "terms": ["Ethnicity[T.Hispanic/Latino]"] * 3,
    },
    {
        "label": "Unknown",
        "indent": True,
        "terms": ["Ethnicity[T.Unknown/Other]"] * 3,
    },
    {
        "label": (
            "Charlson Comorbidity Index "
            "(reference: CCI 0)"
        ),
        "section": True,
    },
    {
        "label": "1–2 (mild)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.1-2 (Mild)]"
        ] * 3,
    },
    {
        "label": "3–4 (moderate)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.3-4 (Moderate)]"
        ] * 3,
    },
    {
        "label": "≥5 (severe)",
        "indent": True,
        "terms": [
            "C(Charlson_group)[T.5+ (Severe)]"
        ] * 3,
    },
    {
        "label": (
            "Patient-initiated messaging "
            "(log, past 90 days)"
        ),
        "terms": [
            obj["x_pinit"] for obj in table_s1_objects
        ],
    },
    {
        "label": (
            "Recent healthcare encounters "
            "(log1p, past 7 days)"
        ),
        "terms": [
            obj["x_encounter"] for obj in table_s1_objects
        ],
    },
    {
        "label": "Message exposures (log, past 90 days)",
        "section": True,
    },
    {
        "label": "System",
        "indent": True,
        "terms": [
            base_obj["x_sys"],
            phys_obj["x_sys"],
            nonphys_obj["x_sys"],
        ],
    },
    {
        "label": "HCW",
        "indent": True,
        "terms": [
            base_obj["x_prov"],
            None,
            None,
        ],
    },
    {
        "label": "HCW × system",
        "indent": True,
        "terms": [
            base_interaction,
            None,
            None,
        ],
    },
    {
        "label": "Physician/APP",
        "indent": True,
        "terms": [
            None,
            phys_obj["x_phys"],
            nonphys_obj["x_phys"],
        ],
    },
    {
        "label": "Other HCW",
        "indent": True,
        "terms": [
            None,
            phys_obj["x_other_human"],
            nonphys_obj["x_other_human"],
        ],
    },
    {
        "label": "Physician/APP × Other HCW",
        "indent": True,
        "terms": [
            None,
            phys_interaction,
            nonphys_interaction,
        ],
    },
]


# ------------------------------------------------------------
# Extract estimates
# ------------------------------------------------------------

table_s1_records = []

for row in table_s1_row_definitions:
    record = {
        "Variable": row["label"],
        "section": bool(row.get("section", False)),
        "indent": bool(row.get("indent", False)),
    }

    if record["section"]:
        estimates = [("", "")] * 3
    else:
        estimates = [
            s1_format_result(model, term)
            for model, term in zip(
                table_s1_models,
                row["terms"],
            )
        ]

    for model_number, (or_ci, p_value) in enumerate(
        estimates,
        start=1,
    ):
        record[
            f"Model {model_number} OR (95% CI)"
        ] = or_ci
        record[
            f"Model {model_number} P value"
        ] = p_value

    table_s1_records.append(record)


table_s1_display = pd.DataFrame(table_s1_records)

table_s1_export = table_s1_display.drop(
    columns=["section", "indent"]
).copy()


# ------------------------------------------------------------
# Title and updated note
# ------------------------------------------------------------

table_s1_title = (
    "Table S1. Sensitivity analysis using a 90-day message "
    "exposure window (patient reply within 7 days)"
)

table_s1_note = (
    "Model covariates included patient age, sex, race, "
    "ethnicity, Charlson Comorbidity Index (CCI), prior "
    "patient-initiated messaging in the preceding 90 days, "
    "and recent healthcare encounters during the 7 days "
    "preceding thread initiation (log[1+count]). Models "
    "incorporated month-year fixed effects to account for "
    "temporal trends with robust standard errors clustered "
    "at the patient level. Reference groups were female "
    "(sex), White (race), non-Hispanic (ethnicity), and CCI "
    "0 (none) for the comorbidity groups. Other race includes "
    "unknown or multiple races, as well as small-cell "
    "categories. Message exposure variables and "
    "patient-initiated messaging were transformed using the "
    "natural logarithm log(1+count). Other HCW indicates "
    "non-physician/APP health care workers. NA indicates a "
    "variable not included in the model. All tests were "
    "2-tailed, with statistical significance defined as "
    "P<.05. Abbreviations: OR, odds ratio; CI, confidence "
    "interval; HCW, health care worker; APP, advanced "
    "practice provider."
)


model_headers = [
    (
        "Reply to any HCW-initiated threads"
        f"<br>(n = {table_s1_sample_sizes[0]:,})"
    ),
    (
        "Reply to physician/APP-initiated threads"
        f"<br>(n = {table_s1_sample_sizes[1]:,})"
    ),
    (
        "Reply to non-physician/APP HCW-initiated threads"
        f"<br>(n = {table_s1_sample_sizes[2]:,})"
    ),
]


# ------------------------------------------------------------
# Build publication-formatted HTML
# ------------------------------------------------------------

body_rows = []

for _, row in table_s1_display.iterrows():
    label = escape(str(row["Variable"]))

    if row["section"]:
        body_rows.append(
            '<tr class="section-row">'
            f"<td>{label}</td>"
            '<td colspan="6"></td>'
            "</tr>"
        )
        continue

    label_class = "indent" if row["indent"] else ""
    cells = [f'<td class="{label_class}">{label}</td>']

    for model_number in range(1, 4):
        or_ci = escape(
            str(
                row[
                    f"Model {model_number} OR (95% CI)"
                ]
            )
        )
        p_value = escape(
            str(
                row[
                    f"Model {model_number} P value"
                ]
            )
        )

        cells.append(
            f'<td class="number">{or_ci}</td>'
        )
        cells.append(
            f'<td class="number">{p_value}</td>'
        )

    body_rows.append(
        "<tr>" + "".join(cells) + "</tr>"
    )


table_s1_html = f"""
<style>
.table-s1-wrap {{
    font-family: "Times New Roman", serif;
    max-width: 1100px;
    color: #000;
}}
.table-s1-title {{
    font-size: 10.5pt;
    font-weight: bold;
    margin-bottom: 6px;
}}
.table-s1 {{
    border-collapse: collapse;
    width: 100%;
    font-size: 9.5pt;
    line-height: 1.05;
}}
.table-s1 th {{
    font-weight: bold;
    text-align: center;
    padding: 3px 4px;
    vertical-align: bottom;
}}
.table-s1 thead tr:first-child th {{
    border-top: 1.2px solid #000;
}}
.table-s1 thead tr:last-child th {{
    border-bottom: 1px solid #000;
}}
.table-s1 td {{
    padding: 2px 4px;
    vertical-align: top;
}}
.table-s1 tbody tr:last-child td {{
    border-bottom: 1.2px solid #000;
}}
.table-s1 .number {{
    text-align: center;
    white-space: nowrap;
}}
.table-s1 .indent {{
    padding-left: 18px;
}}
.table-s1 .section-row td:first-child {{
    font-weight: bold;
    padding-top: 5px;
}}
.table-s1-note {{
    font-size: 9pt;
    line-height: 1.15;
    margin-top: 8px;
    text-align: left;
}}
</style>

<div class="table-s1-wrap">
  <div class="table-s1-title">
    {escape(table_s1_title)}
  </div>

  <table class="table-s1">
    <thead>
      <tr>
        <th rowspan="2" style="text-align:left">
          Variable
        </th>
        <th colspan="2">{model_headers[0]}</th>
        <th colspan="2">{model_headers[1]}</th>
        <th colspan="2">{model_headers[2]}</th>
      </tr>
      <tr>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
        <th>OR (95% CI)</th>
        <th><i>P</i> value</th>
      </tr>
    </thead>

    <tbody>
      {''.join(body_rows)}
    </tbody>
  </table>

  <div class="table-s1-note">
    {escape(table_s1_note)}
  </div>
</div>
"""


display(HTML(table_s1_html))


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

table_s1_csv_path = (
    TABLE_DIR
    / "Table_S1_encounter_adjusted_90d.csv"
)

table_s1_html_path = (
    TABLE_DIR
    / "Table_S1_encounter_adjusted_90d.html"
)

table_s1_note_path = (
    TABLE_DIR
    / "Table_S1_encounter_adjusted_90d_note.txt"
)

table_s1_export.to_csv(
    table_s1_csv_path,
    index=False,
    encoding="utf-8-sig",
)

table_s1_html_path.write_text(
    table_s1_html,
    encoding="utf-8",
)

table_s1_note_path.write_text(
    table_s1_title + "\n\n" + table_s1_note,
    encoding="utf-8",
)

print("Saved Table S1:")
for path in [
    table_s1_csv_path,
    table_s1_html_path,
    table_s1_note_path,
]:
    print(" -", path)

## 9. Sensitivity Analyses: Tables S2 and S3

Table S2 uses patient reply within 30 days as the outcome. Table S3 retains the 7-day reply outcome and excludes thread initiations from calendar year 2021. Both analyses use the 30-day message-exposure window and the revised encounter-adjusted model specification.

In [ ]:
# Encounter-adjusted sensitivity models for Tables S2 and S3
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def add_reply_30d_outcome(data: pd.DataFrame) -> pd.DataFrame:
    """Create the 30-day reply outcome from the first reply timestamp."""
    out = data.copy()
    if "patient_replied_30d" not in out.columns:
        required = {"first_patient_reply_time", "thread_first_time"}
        missing = required - set(out.columns)
        if missing:
            raise KeyError(
                "Cannot create patient_replied_30d; missing: "
                + ", ".join(sorted(missing))
            )
        reply_time = pd.to_datetime(out["first_patient_reply_time"], errors="coerce")
        thread_time = pd.to_datetime(out["thread_first_time"], errors="coerce")
        out["patient_replied_30d"] = (
            reply_time.notna()
            & (reply_time >= thread_time)
            & (reply_time <= thread_time + pd.Timedelta(days=30))
        ).astype(int)
    return out


def fit_encounter_adjusted_sensitivity_model(
    data: pd.DataFrame,
    outcome: str,
    model_group: str,
    period: str = "30d",
) -> dict:
    """Fit one revised encounter-adjusted sensitivity model."""
    df = data.copy()
    df[outcome] = pd.to_numeric(df[outcome], errors="coerce")

    x_system = f"log_system_msgs_past_{period}"
    x_pinit = f"log_patient_initiated_threads_past_{period}"
    encounter = "log_encounters_past_7d"

    if model_group == "any_hcw":
        x_hcw = f"log_provider_msgs_past_{period}"
        interaction = f"{x_hcw}:{x_system}"
        exposure_terms = [x_hcw, x_system, interaction]
        exposure_columns = [x_hcw, x_system]
        term_map = {
            "system": x_system,
            "hcw": x_hcw,
            "hcw_system": interaction,
            "physapp": None,
            "other_hcw": None,
            "physapp_other": None,
        }
    elif model_group in {"physapp", "nonphysapp"}:
        x_physapp = f"log_physapp_msgs_past_{period}"
        x_other = f"log_other_human_msgs_past_{period}"
        interaction = f"{x_physapp}:{x_other}"
        exposure_terms = [x_physapp, x_other, x_system, interaction]
        exposure_columns = [x_physapp, x_other, x_system]
        term_map = {
            "system": x_system,
            "hcw": None,
            "hcw_system": None,
            "physapp": x_physapp,
            "other_hcw": x_other,
            "physapp_other": interaction,
        }
    else:
        raise ValueError(f"Unknown model_group: {model_group}")

    required_columns = [
        outcome, "DMRN", "month_year", "Age_at_Message",
        "Gender", "Race", "Ethnicity", "Charlson_group",
        encounter, x_pinit, *exposure_columns,
    ]
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise KeyError("Missing model columns: " + ", ".join(missing_columns))

    df_fit = df.dropna(subset=required_columns).copy()
    df_fit[outcome] = df_fit[outcome].astype(int)

    model_terms = (
        exposure_terms
        + [x_pinit, encounter, "Age_at_Message", "C(Gender)"]
        + ['C(Race, Treatment(reference="White"))']
        + ["Ethnicity", "C(Charlson_group)", "C(month_year)"]
    )
    formula = f"{outcome} ~ " + " + ".join(model_terms)
    model = smf.logit(formula, data=df_fit).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_fit["DMRN"]},
        disp=False,
    )

    return {
        "model": model,
        "N": int(len(df_fit)),
        "outcome": outcome,
        "period": period,
        "model_group": model_group,
        "x_pinit": x_pinit,
        "x_encounter": encounter,
        **term_map,
    }


def fit_sensitivity_suite(sources, outcome):
    groups = ["any_hcw", "physapp", "nonphysapp"]
    return [
        fit_encounter_adjusted_sensitivity_model(source, outcome, group)
        for source, group in zip(sources, groups)
    ]


# Table S2: 30-day reply outcome, full analytic period.
s2_sources = [
    add_reply_30d_outcome(filtered_df),
    add_reply_30d_outcome(analysis_physapp),
    add_reply_30d_outcome(analysis_other),
]
MODELS_TABLE_S2 = fit_sensitivity_suite(s2_sources, "patient_replied_30d")

# Table S3: 7-day reply outcome after excluding 2021 anchor threads.
s3_sources = [
    source.loc[~source["month_year"].astype(str).str.startswith("2021")].copy()
    for source in [filtered_df, analysis_physapp, analysis_other]
]
MODELS_TABLE_S3 = fit_sensitivity_suite(s3_sources, "patient_replied_7d")

sensitivity_sample_summary = pd.DataFrame({
    "Analysis": ["Table S2: reply within 30 days", "Table S3: excluding 2021"],
    "Any HCW N": [MODELS_TABLE_S2[0]["N"], MODELS_TABLE_S3[0]["N"]],
    "Physician/APP N": [MODELS_TABLE_S2[1]["N"], MODELS_TABLE_S3[1]["N"]],
    "Non-physician/APP N": [MODELS_TABLE_S2[2]["N"], MODELS_TABLE_S3[2]["N"]],
})
display(sensitivity_sample_summary)


In [ ]:
# Publication-formatted Tables S2 and S3
# ============================================================

from html import escape
from IPython.display import HTML, display


def sensitivity_or_ci_p(model, term):
    if term is None:
        return "NA", "NA"
    beta = float(model.params[term])
    interval = model.conf_int().loc[term].astype(float)
    odds_ratio = float(np.exp(beta))
    lower = float(np.exp(interval.iloc[0]))
    upper = float(np.exp(interval.iloc[1]))
    p_value = float(model.pvalues[term])
    or_ci = f"{odds_ratio:.2f} ({lower:.2f}\u2013{upper:.2f})"
    if p_value < 0.001:
        p_text = "<.001"
    elif p_value < 0.01:
        p_text = f"{p_value:.3f}".lstrip("0")
    else:
        p_text = f"{p_value:.2f}".lstrip("0")
    return or_ci, p_text


def sensitivity_row_definitions(model_objects):
    race_prefix = 'C(Race, Treatment(reference="White"))'
    return [
        {"label": "Age at message (year)", "terms": ["Age_at_Message"] * 3},
        {"label": "Sex (reference: female)", "section": True},
        {"label": "Male", "indent": True, "terms": ["C(Gender)[T.Male]"] * 3},
        {"label": "Race (reference: White)", "section": True},
        {"label": "Asian", "indent": True, "terms": [f'{race_prefix}[T.Asian]'] * 3},
        {"label": "Black or African American", "indent": True, "terms": [f'{race_prefix}[T.Black/African American]'] * 3},
        {"label": "Other", "indent": True, "terms": [f'{race_prefix}[T.Other/Multiple/Unknown]'] * 3},
        {"label": "Ethnicity (reference: non-Hispanic)", "section": True},
        {"label": "Hispanic/Latino", "indent": True, "terms": ["Ethnicity[T.Hispanic/Latino]"] * 3},
        {"label": "Unknown", "indent": True, "terms": ["Ethnicity[T.Unknown/Other]"] * 3},
        {"label": "Charlson Comorbidity Index (reference: CCI 0)", "section": True},
        {"label": "1\u20132 (mild)", "indent": True, "terms": ["C(Charlson_group)[T.1-2 (Mild)]"] * 3},
        {"label": "3\u20134 (moderate)", "indent": True, "terms": ["C(Charlson_group)[T.3-4 (Moderate)]"] * 3},
        {"label": "\u22655 (severe)", "indent": True, "terms": ["C(Charlson_group)[T.5+ (Severe)]"] * 3},
        {"label": "Patient-initiated messaging (log, past 30 days)", "terms": [obj["x_pinit"] for obj in model_objects]},
        {"label": "Recent healthcare encounters (log1p, past 7 days)", "terms": [obj["x_encounter"] for obj in model_objects]},
        {"label": "Message exposures (log, past 30 days)", "section": True},
        {"label": "System", "indent": True, "terms": [obj["system"] for obj in model_objects]},
        {"label": "HCW", "indent": True, "terms": [model_objects[0]["hcw"], None, None]},
        {"label": "HCW \u00d7 system", "indent": True, "terms": [model_objects[0]["hcw_system"], None, None]},
        {"label": "Physician/APP", "indent": True, "terms": [None, model_objects[1]["physapp"], model_objects[2]["physapp"]]},
        {"label": "Other HCW", "indent": True, "terms": [None, model_objects[1]["other_hcw"], model_objects[2]["other_hcw"]]},
        {"label": "Physician/APP \u00d7 Other HCW", "indent": True, "terms": [None, model_objects[1]["physapp_other"], model_objects[2]["physapp_other"]]},
    ]


def render_sensitivity_table(model_objects, title, note, file_stem):
    models = [obj["model"] for obj in model_objects]
    sample_sizes = [int(obj["N"]) for obj in model_objects]
    records = []
    for row in sensitivity_row_definitions(model_objects):
        record = {
            "Variable": row["label"],
            "section": bool(row.get("section", False)),
            "indent": bool(row.get("indent", False)),
        }
        estimates = (
            [("", "")] * 3
            if record["section"]
            else [
                sensitivity_or_ci_p(model, term)
                for model, term in zip(models, row["terms"])
            ]
        )
        for number, (or_ci, p_value) in enumerate(estimates, start=1):
            record[f"Model {number} OR (95% CI)"] = or_ci
            record[f"Model {number} P value"] = p_value
        records.append(record)

    display_df = pd.DataFrame(records)
    export_df = display_df.drop(columns=["section", "indent"])
    headers = [
        f"Reply to any HCW-initiated threads<br>(n = {sample_sizes[0]:,})",
        f"Reply to physician/APP-initiated threads<br>(n = {sample_sizes[1]:,})",
        f"Reply to non-physician/APP HCW-initiated threads<br>(n = {sample_sizes[2]:,})",
    ]

    body_rows = []
    for _, row in display_df.iterrows():
        label = escape(str(row["Variable"]))
        if row["section"]:
            body_rows.append(f'<tr class="section-row"><td>{label}</td><td colspan="6"></td></tr>')
            continue
        label_class = "indent" if row["indent"] else ""
        cells = [f'<td class="{label_class}">{label}</td>']
        for number in range(1, 4):
            cells.append(f'<td class="number">{escape(str(row[f"Model {number} OR (95% CI)"]))}</td>')
            cells.append(f'<td class="number">{escape(str(row[f"Model {number} P value"]))}</td>')
        body_rows.append("<tr>" + "".join(cells) + "</tr>")

    html = f"""
    <style>
    .sensitivity-wrap {{font-family: 'Times New Roman', serif; max-width: 1100px; color: #000;}}
    .sensitivity-title {{font-size: 10.5pt; font-weight: bold; margin-bottom: 6px;}}
    .sensitivity-table {{border-collapse: collapse; width: 100%; font-size: 9.5pt; line-height: 1.05;}}
    .sensitivity-table th {{font-weight: bold; text-align: center; padding: 3px 4px; vertical-align: bottom;}}
    .sensitivity-table thead tr:first-child th {{border-top: 1.2px solid #000;}}
    .sensitivity-table thead tr:last-child th {{border-bottom: 1px solid #000;}}
    .sensitivity-table td {{padding: 2px 4px; vertical-align: top;}}
    .sensitivity-table tbody tr:last-child td {{border-bottom: 1.2px solid #000;}}
    .sensitivity-table .number {{text-align: center; white-space: nowrap;}}
    .sensitivity-table .indent {{padding-left: 18px;}}
    .sensitivity-table .section-row td:first-child {{font-weight: bold; padding-top: 5px;}}
    .sensitivity-note {{font-size: 9pt; line-height: 1.15; margin-top: 8px; text-align: left;}}
    </style>
    <div class="sensitivity-wrap">
      <div class="sensitivity-title">{escape(title)}</div>
      <table class="sensitivity-table">
        <thead>
          <tr><th rowspan="2" style="text-align:left">Variable</th>
              <th colspan="2">{headers[0]}</th><th colspan="2">{headers[1]}</th><th colspan="2">{headers[2]}</th></tr>
          <tr><th>OR (95% CI)</th><th><i>P</i> value</th><th>OR (95% CI)</th><th><i>P</i> value</th><th>OR (95% CI)</th><th><i>P</i> value</th></tr>
        </thead>
        <tbody>{''.join(body_rows)}</tbody>
      </table>
      <div class="sensitivity-note">{escape(note)}</div>
    </div>
    """

    csv_path = TABLE_DIR / f"{file_stem}.csv"
    html_path = TABLE_DIR / f"{file_stem}.html"
    note_path = TABLE_DIR / f"{file_stem}_note.txt"
    export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    html_path.write_text(html, encoding="utf-8")
    note_path.write_text(title + "\n\n" + note, encoding="utf-8")
    display(HTML(html))
    print("Saved:", csv_path, html_path, note_path, sep="\n - " )
    return export_df


common_note = (
    "Model covariates included patient age, sex, race, ethnicity, Charlson "
    "Comorbidity Index (CCI), prior patient-initiated messaging in the preceding "
    "30 days, and recent healthcare encounters during the 7 days preceding thread "
    "initiation (log[1+count]). Models incorporated month-year fixed effects to "
    "account for temporal trends with robust standard errors clustered at the "
    "patient level. Reference groups were female (sex), White (race), non-Hispanic "
    "(ethnicity), and CCI 0 (none). Other race includes unknown or multiple races, "
    "as well as small-cell categories. Message exposure variables and patient-initiated "
    "messaging were transformed using log(1+count). Other HCW indicates non-physician/APP "
    "health care workers. NA indicates a variable not included in the model. All tests "
    "were 2-tailed, with statistical significance defined as P<.05. Abbreviations: "
    "OR, odds ratio; CI, confidence interval; HCW, health care worker; APP, advanced "
    "practice provider."
)

table_s2 = render_sensitivity_table(
    MODELS_TABLE_S2,
    "Table S2. Sensitivity analysis using patient reply within 30 days as the outcome",
    common_note,
    "Table_S2_reply_within_30d_encounter_adjusted",
)

table_s3 = render_sensitivity_table(
    MODELS_TABLE_S3,
    "Table S3. Sensitivity analysis excluding messages from 2021 (patient reply within 7 days)",
    "Models were restricted to thread initiations in calendar years 2022 and 2023. " + common_note,
    "Table_S3_excluding_2021_encounter_adjusted",
)


## 10. Figure 1 Interaction Panels

The saved filenames include `encounter7d_adjusted` so they do not overwrite older figures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ============================================================
# Figure 1 helper: full-row marginal predictions
# ============================================================
# This section intentionally uses the fitted model objects already created
# above. It does not refit models or alter samples, coefficients, exposure
# definitions, or interaction percentiles.

PREFERRED_CATEGORY_VALUES = {
    "Gender": ["Female"],
    "Race": ["White"],
    "Ethnicity": [
        "Not Hispanic/Latino",
        "Non-Hispanic",
        "Not Hispanic or Latino",
        "Not Hispanic",
    ],
    "Charlson_group": ["0 (None)", "0", "None"],
}

STANDARD_COVARIATES = [
    "Age_at_Message",
    "Gender",
    "Race",
    "Ethnicity",
    "Charlson_group",
    "month_year",
]


def _nice_xticks(max_x, step=2):
    max_xi = int(np.ceil(max_x))
    return np.arange(0, max_xi + 1, step)


def _format_profile_value(value):
    if pd.isna(value):
        return "NA"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6g}"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return str(value)


def _numeric_median(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.nan
    return float(values.median())


def _mode_value(series):
    values = series.dropna()
    if values.empty:
        return np.nan
    mode_values = values.mode(dropna=True)
    if not mode_values.empty:
        return mode_values.iloc[0]
    return values.iloc[0]


def _choose_category_value(df, column):
    values = df[column].dropna()
    if values.empty:
        return np.nan, "missing"

    preferred_values = PREFERRED_CATEGORY_VALUES.get(column, [])
    unique_values = list(values.unique())
    unique_as_text = {str(value): value for value in unique_values}

    for preferred in preferred_values:
        if preferred in unique_as_text:
            return unique_as_text[preferred], "reference"

    return _mode_value(values), "modal"


def _candidate_prediction_columns(obj, df, x_log_name, m_log_name):
    columns = []

    for column in STANDARD_COVARIATES:
        if column in df.columns:
            columns.append(column)

    for key, value in obj.items():
        if key.startswith("x_") and isinstance(value, str) and value in df.columns:
            columns.append(value)

    for column in (x_log_name, m_log_name):
        if column in df.columns:
            columns.append(column)

    return list(dict.fromkeys(columns))


def _build_representative_profile(df, obj, x_log_name, m_log_name):
    profile = {}
    records = []

    for column in _candidate_prediction_columns(obj, df, x_log_name, m_log_name):
        if column in (x_log_name, m_log_name):
            value = _numeric_median(df[column])
            basis = "varied in figure"
        elif pd.api.types.is_numeric_dtype(df[column]):
            value = _numeric_median(df[column])
            basis = "sample median"
        else:
            value, basis = _choose_category_value(df, column)

        profile[column] = value
        records.append({
            "covariate": column,
            "value": value,
            "display_value": _format_profile_value(value),
            "basis": basis,
        })

    return profile, records


def _panel_from_registry(store, key, df_for_plot, x_cnt_col, m_cnt_col, x_log_name, m_log_name):
    if key not in store:
        raise KeyError(f"Key '{key}' not found. Available keys: {list(store.keys())[:10]} ...")

    obj = store[key]
    df_use = obj.get("df_fit", df_for_plot)

    missing = [col for col in [x_cnt_col, m_cnt_col, x_log_name, m_log_name] if col not in df_use.columns]
    if missing:
        raise ValueError(f"{key} prediction dataframe is missing columns: {missing}")

    profile, profile_records = _build_representative_profile(
        df_use,
        obj,
        x_log_name=x_log_name,
        m_log_name=m_log_name,
    )

    return {
        "key": key,
        "obj": obj,
        "model": obj["model"],
        "df": df_use,
        "x_cnt_col": x_cnt_col,
        "m_cnt_col": m_cnt_col,
        "x_log_name": x_log_name,
        "m_log_name": m_log_name,
        "profile": profile,
        "profile_records": profile_records,
    }


def _percentile_count(series, percentile):
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if values.size == 0:
        raise ValueError("Cannot compute percentile from an empty series.")
    return max(float(np.percentile(values, percentile)), 0.0)


def _x_axis_counts(df, x_cnt_col, x_cap_quantile, n_points):
    if x_cap_quantile is None:
        max_x = float(pd.to_numeric(df[x_cnt_col], errors="coerce").max())
    else:
        max_x = float(np.floor(pd.to_numeric(df[x_cnt_col], errors="coerce").quantile(x_cap_quantile)))
    max_x = max(max_x, 1.0)
    return np.linspace(0, max_x, n_points), max_x


def _prediction_dataframe(panel, x_counts, m_count):
    rows = []
    for x_count in np.asarray(x_counts, dtype=float):
        row = dict(panel["profile"])
        row[panel["x_log_name"]] = float(np.log1p(x_count))
        row[panel["m_log_name"]] = float(np.log1p(float(m_count)))
        rows.append(row)
    return pd.DataFrame(rows)


def _predict_probabilities_full_rows(panel, x_counts, m_count):
    pred_df = _prediction_dataframe(panel, x_counts, m_count)
    return panel["model"].predict(pred_df).astype(float).to_numpy()


def _predict_probability_single(panel, x_count, m_count):
    return float(_predict_probabilities_full_rows(panel, [x_count], m_count)[0])


def predict_prob_curves_model_predict(
    panel,
    x_cap_quantile=0.99,
    percentiles=(50, 90, 99),
    n_points=250,
):
    x_counts, max_x = _x_axis_counts(
        panel["df"],
        panel["x_cnt_col"],
        x_cap_quantile=x_cap_quantile,
        n_points=n_points,
    )

    m_vals = {
        pct: _percentile_count(panel["df"][panel["m_cnt_col"]], pct)
        for pct in percentiles
    }

    preds = {
        pct: _predict_probabilities_full_rows(panel, x_counts, m_vals[pct])
        for pct in percentiles
    }

    return x_counts, preds, m_vals, max_x


def add_panel_header_oneline(ax, letter, title, x=0.00, y=1.06, pad=0.012):
    t_letter = ax.text(
        x,
        y,
        letter,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        fontweight="bold",
        bbox=dict(
            boxstyle="square,pad=0.18",
            facecolor="white",
            edgecolor="black",
            linewidth=0.8,
        ),
        clip_on=False,
    )

    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    bb = t_letter.get_window_extent(renderer=renderer)

    x0_px = ax.transAxes.transform((0, 0))[0]
    x1_px = ax.transAxes.transform((1, 0))[0]
    axes_w_px = x1_px - x0_px
    box_w_axes = bb.width / axes_w_px

    ax.text(
        x + box_w_axes + pad,
        y,
        title,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        clip_on=False,
    )


def _collect_profile_rows(period, panel_letter, panel_title, panel, m_vals):
    rows = []
    for record in panel["profile_records"]:
        basis = record["basis"]
        if record["covariate"] == panel["x_log_name"]:
            basis = "varied on x-axis"
        elif record["covariate"] == panel["m_log_name"]:
            basis = "displayed at 50th, 90th, and 99th percentiles"

        rows.append({
            "period": period,
            "panel": panel_letter,
            "panel_title": panel_title,
            "model_key": panel["key"],
            "covariate": record["covariate"],
            "value": record["display_value"],
            "basis": basis,
        })

    rows.append({
        "period": period,
        "panel": panel_letter,
        "panel_title": panel_title,
        "model_key": panel["key"],
        "covariate": panel["m_cnt_col"],
        "value": "; ".join(f"p{pct}={m_vals[pct]:.0f}" for pct in sorted(m_vals)),
        "basis": "displayed interacting-exposure counts",
    })

    return rows


def _collect_selected_prediction_rows(
    period,
    panel_letter,
    panel_title,
    panel,
    m_vals,
    max_x,
    selected_focal_counts=(0, 5, 10),
):
    rows = []
    for focal_count in selected_focal_counts:
        for pct in sorted(m_vals):
            probability = _predict_probability_single(panel, focal_count, m_vals[pct])
            rows.append({
                "period": period,
                "panel": panel_letter,
                "panel_title": panel_title,
                "model_key": panel["key"],
                "focal_exposure": panel["x_cnt_col"],
                "focal_exposure_count": float(focal_count),
                "within_plotted_x_range": bool(float(focal_count) <= float(max_x)),
                "interacting_exposure": panel["m_cnt_col"],
                "interacting_exposure_percentile": int(pct),
                "interacting_exposure_count": float(m_vals[pct]),
                "predicted_reply_probability": probability,
                "predicted_reply_probability_pct": 100.0 * probability,
            })
    return rows


def _auto_ylim(panel_results, minimum_upper=30.0):
    max_prob = 0.0
    for result in panel_results.values():
        for values in result["preds"].values():
            max_prob = max(max_prob, float(np.nanmax(values)) * 100.0)
    upper = max(minimum_upper, float(np.ceil((max_prob + 2.5) / 5.0) * 5.0))
    return 0.0, upper


def plot_interaction_panels_3_compact(
    period,
    panelA,
    panelB,
    panelC,
    A_title="HCW-Generated x System-Generated Messages",
    B_title="Physician/APP x Other HCW + System Messages",
    C_title="Physician/APP x Other HCW + System Messages",
    x_cap_quantile=0.99,
    percentiles=(50, 90, 99),
    n_points=250,
    y_lim=None,
    y_label="Predicted probability\nof patient reply 7d (%)",
    A_legend_title="System message percentile (messages)",
    B_legend_title="Other message percentile (messages)",
    C_legend_title="Other message percentile (messages)",
    A_xlabel=None,
    B_xlabel=None,
    C_xlabel=None,
    outpdf=None,
    outpng=None,
    selected_focal_counts=(0, 5, 10),
):
    plt.rcParams.update({
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "legend.title_fontsize": 9,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": ":",
        "savefig.dpi": 300,
    })

    panel_specs = [
        ("A", panelA, A_title, A_legend_title, A_xlabel),
        ("B", panelB, B_title, B_legend_title, B_xlabel),
        ("C", panelC, C_title, C_legend_title, C_xlabel),
    ]

    panel_results = {}
    for letter, panel, title, legend_title, xlabel in panel_specs:
        x_counts, preds, m_vals, max_x = predict_prob_curves_model_predict(
            panel,
            x_cap_quantile=x_cap_quantile,
            percentiles=percentiles,
            n_points=n_points,
        )
        panel_results[letter] = {
            "panel": panel,
            "title": title,
            "legend_title": legend_title,
            "xlabel": xlabel,
            "x_counts": x_counts,
            "preds": preds,
            "m_vals": m_vals,
            "max_x": max_x,
        }

    if y_lim is None:
        y_lim = _auto_ylim(panel_results)

    style_map = {
        50: {"color": "black", "linestyle": "-", "marker": "o", "label": "50th (median)"},
        90: {"color": "dimgray", "linestyle": "--", "marker": "s", "label": "90th"},
        99: {"color": "gray", "linestyle": "-.", "marker": "^", "label": "99th"},
    }

    fig, axes = plt.subplots(1, 3, figsize=(17.2, 4.4), sharey=True)

    profile_rows = []
    prediction_rows = []

    for ax, (letter, result) in zip(axes, panel_results.items()):
        for pct in percentiles:
            st = style_map[pct]
            ax.plot(
                result["x_counts"],
                result["preds"][pct] * 100.0,
                lw=2.1,
                color=st["color"],
                linestyle=st["linestyle"],
                marker=st["marker"],
                markersize=4.8,
                markevery=20,
                label=f"{st['label']} ({result['m_vals'][pct]:.0f} msgs)",
            )

        add_panel_header_oneline(ax, letter, result["title"], x=-0.03, y=1.15, pad=0.030)

        xlabel = result["xlabel"]
        if xlabel is None:
            xlabel = f"Message volume (past {period}; count; capped at p99)"
        ax.set_xlabel(xlabel)
        ax.set_xlim(0, result["max_x"])
        ax.set_xticks(_nice_xticks(result["max_x"], step=2))
        ax.legend(
            frameon=False,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.22),
            ncol=3,
            title=result["legend_title"],
        )

        profile_rows.extend(_collect_profile_rows(
            period,
            letter,
            result["title"],
            result["panel"],
            result["m_vals"],
        ))
        prediction_rows.extend(_collect_selected_prediction_rows(
            period,
            letter,
            result["title"],
            result["panel"],
            result["m_vals"],
            result["max_x"],
            selected_focal_counts=selected_focal_counts,
        ))

    for ax in axes:
        ax.set_ylabel(y_label)
        ax.set_ylim(*y_lim)
        ax.set_yticks(np.arange(y_lim[0], y_lim[1] + 0.001, 10.0))

    for ax in axes[1:]:
        ax.tick_params(axis="y", which="both", labelleft=True)

    plt.subplots_adjust(left=0.06, right=0.99, top=0.83, bottom=0.28, wspace=0.26)

    if outpdf:
        fig.savefig(outpdf, format="pdf", bbox_inches="tight", pad_inches=0.05)
        print(f"Saved vector PDF: {outpdf}")

    if outpng:
        png_dpi = 300
        fig.savefig(
            outpng,
            format="png",
            dpi=png_dpi,
            bbox_inches="tight",
            pad_inches=0.05,
            facecolor="white",
            edgecolor="white",
        )
        print(f"Saved PNG: {outpng} (dpi={png_dpi})")

    plt.show()

    return {
        "period": period,
        "pdf_path": outpdf,
        "png_path": outpng,
        "y_lim": y_lim,
        "panel_results": panel_results,
        "profile_rows": profile_rows,
        "prediction_rows": prediction_rows,
    }


# ============================================================
# Calls: 30d and 90d Figure 1 panels
# ============================================================

dfA_plot = filtered_df.copy()
dfB_plot = analysis_physapp.copy()
dfC_plot = analysis_other.copy()

MODELS_FIG1 = {}
MODELS_FIG1.update(MODELS_BASE)
MODELS_FIG1.update(MODELS_PHYS)
MODELS_FIG1.update(MODELS_OTHER)

print("Combined keys:", sorted(MODELS_FIG1.keys()))

panelA_30d = _panel_from_registry(
    MODELS_FIG1,
    "BASE_HCWxSYS_30d",
    df_for_plot=dfA_plot,
    x_cnt_col="provider_msgs_past_30d",
    m_cnt_col="system_msgs_past_30d",
    x_log_name="log_provider_msgs_past_30d",
    m_log_name="log_system_msgs_past_30d",
)

panelB_30d = _panel_from_registry(
    MODELS_FIG1,
    "FOC_PHYSINIT_30d",
    df_for_plot=dfB_plot,
    x_cnt_col="physapp_msgs_past_30d",
    m_cnt_col="other_human_msgs_past_30d",
    x_log_name="log_physapp_msgs_past_30d",
    m_log_name="log_other_human_msgs_past_30d",
)

panelC_30d = _panel_from_registry(
    MODELS_FIG1,
    "FOC_NONPHYSINIT_30d",
    df_for_plot=dfC_plot,
    x_cnt_col="physapp_msgs_past_30d",
    m_cnt_col="other_human_msgs_past_30d",
    x_log_name="log_physapp_msgs_past_30d",
    m_log_name="log_other_human_msgs_past_30d",
)

FIGURE1_RESULTS = []
FIGURE1_RESULTS.append(plot_interaction_panels_3_compact(
    period="30d",
    panelA=panelA_30d,
    panelB=panelB_30d,
    panelC=panelC_30d,
    A_title="HCW x System\nReply to all HCW threads",
    B_title="Physician/APP x Other HCW\nReply to physician/APP-initiated threads",
    C_title="Physician/APP x Other HCW\nReply to other HCW-initiated threads",
    A_xlabel="HCW message volume (past 30 days; count; capped at p99)",
    B_xlabel="Physician/APP message volume (past 30 days; count; capped at p99)",
    C_xlabel="Physician/APP message volume (past 30 days; count; capped at p99)",
    A_legend_title="System message percentile (messages)",
    B_legend_title="Other HCW message percentile (messages)",
    C_legend_title="Other HCW message percentile (messages)",
    y_lim=None,
    outpdf=FIGURE_DIR / "Figure1_interaction_panels_30d_3panel_encounter7d_adjusted.pdf",
    outpng=FIGURE_DIR / "Figure1_interaction_panels_30d_3panel_encounter7d_adjusted.png",
))

panelA_90d = _panel_from_registry(
    MODELS_FIG1,
    "BASE_HCWxSYS_90d",
    df_for_plot=dfA_plot,
    x_cnt_col="provider_msgs_past_90d",
    m_cnt_col="system_msgs_past_90d",
    x_log_name="log_provider_msgs_past_90d",
    m_log_name="log_system_msgs_past_90d",
)

panelB_90d = _panel_from_registry(
    MODELS_FIG1,
    "FOC_PHYSINIT_90d",
    df_for_plot=dfB_plot,
    x_cnt_col="physapp_msgs_past_90d",
    m_cnt_col="other_human_msgs_past_90d",
    x_log_name="log_physapp_msgs_past_90d",
    m_log_name="log_other_human_msgs_past_90d",
)

panelC_90d = _panel_from_registry(
    MODELS_FIG1,
    "FOC_NONPHYSINIT_90d",
    df_for_plot=dfC_plot,
    x_cnt_col="physapp_msgs_past_90d",
    m_cnt_col="other_human_msgs_past_90d",
    x_log_name="log_physapp_msgs_past_90d",
    m_log_name="log_other_human_msgs_past_90d",
)

FIGURE1_RESULTS.append(plot_interaction_panels_3_compact(
    period="90d",
    panelA=panelA_90d,
    panelB=panelB_90d,
    panelC=panelC_90d,
    A_title="HCW x System\nReply to all HCW threads",
    B_title="Physician/APP x Other HCW\nReply to physician/APP-initiated threads",
    C_title="Physician/APP x Other HCW\nReply to other HCW-initiated threads",
    A_xlabel="HCW message volume (past 90 days; count; capped at p99)",
    B_xlabel="Physician/APP message volume (past 90 days; count; capped at p99)",
    C_xlabel="Physician/APP message volume (past 90 days; count; capped at p99)",
    A_legend_title="System message percentile (messages)",
    B_legend_title="Other HCW message percentile (messages)",
    C_legend_title="Other HCW message percentile (messages)",
    y_lim=None,
    outpdf=FIGURE_DIR / "Figure1_interaction_panels_90d_3panel_encounter7d_adjusted.pdf",
    outpng=FIGURE_DIR / "Figure1_interaction_panels_90d_3panel_encounter7d_adjusted.png",
))

FIGURE1_PROFILE_TABLE = pd.DataFrame(
    row for result in FIGURE1_RESULTS for row in result["profile_rows"]
)
FIGURE1_PREDICTION_TABLE = pd.DataFrame(
    row for result in FIGURE1_RESULTS for row in result["prediction_rows"]
)
FIGURE1_EXPORTS = pd.DataFrame([{
    "period": result["period"],
    "png_path": str(result["png_path"]),
    "pdf_path": str(result["pdf_path"]),
    "y_axis_lower": result["y_lim"][0],
    "y_axis_upper": result["y_lim"][1],
} for result in FIGURE1_RESULTS])

figure1_profile_path = TABLE_DIR / "figure1_representative_covariate_values.csv"
figure1_prediction_path = TABLE_DIR / "figure1_corrected_predicted_probabilities_selected.csv"
figure1_exports_path = TABLE_DIR / "figure1_export_paths.csv"

FIGURE1_PROFILE_TABLE.to_csv(figure1_profile_path, index=False)
FIGURE1_PREDICTION_TABLE.to_csv(figure1_prediction_path, index=False)
FIGURE1_EXPORTS.to_csv(figure1_exports_path, index=False)

print("Saved Figure 1 representative covariates:", figure1_profile_path)
print("Saved Figure 1 selected predicted probabilities:", figure1_prediction_path)
print("Saved Figure 1 export paths:", figure1_exports_path)

display(FIGURE1_PROFILE_TABLE)
display(FIGURE1_PREDICTION_TABLE)
display(FIGURE1_EXPORTS)


## Absolute Predicted-Probability Contrasts

In [ ]:
# Absolute Predicted-Probability Contrasts
# Practical vs statistical significance for Table 3 / Figure 1 models
#
# Placement: run after the primary 30-day models and the corrected Figure 1
# helper cell. This code does not refit models and does not generate figures.
# It reuses the same full model.predict(pd.DataFrame(...)) prediction-row
# framework used for the corrected Figure 1 panels.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------
# Primary 30-day Table 3 panels
# ----------------------------
panel_absolute_A = _panel_from_registry(
    MODELS_BASE,
    "BASE_HCWxSYS_30d",
    df_for_plot=filtered_df,
    x_cnt_col="provider_msgs_past_30d",
    m_cnt_col="system_msgs_past_30d",
    x_log_name="log_provider_msgs_past_30d",
    m_log_name="log_system_msgs_past_30d",
)

panel_absolute_B = _panel_from_registry(
    MODELS_PHYS,
    "FOC_PHYSINIT_30d",
    df_for_plot=analysis_physapp,
    x_cnt_col="physapp_msgs_past_30d",
    m_cnt_col="other_human_msgs_past_30d",
    x_log_name="log_physapp_msgs_past_30d",
    m_log_name="log_other_human_msgs_past_30d",
)

panel_absolute_C = _panel_from_registry(
    MODELS_OTHER,
    "FOC_NONPHYSINIT_30d",
    df_for_plot=analysis_other,
    x_cnt_col="physapp_msgs_past_30d",
    m_cnt_col="other_human_msgs_past_30d",
    x_log_name="log_physapp_msgs_past_30d",
    m_log_name="log_other_human_msgs_past_30d",
)

ABSOLUTE_PANEL_SPECS = [
    (
        "A",
        panel_absolute_A,
        "Reply to any HCW-initiated threads",
        "HCW message volume, past 30 days",
        "System-message volume, past 30 days",
    ),
    (
        "B",
        panel_absolute_B,
        "Reply to physician/APP-initiated threads",
        "Physician/APP message volume, past 30 days",
        "Other-HCW message volume, past 30 days",
    ),
    (
        "C",
        panel_absolute_C,
        "Reply to non-physician/APP HCW-initiated threads",
        "Physician/APP message volume, past 30 days",
        "Other-HCW message volume, past 30 days",
    ),
]


def _absolute_contrasts(panel, panel_letter, model_label, focal_label, modifier_label):
    df = panel["df"]

    x50 = _percentile_count(df[panel["x_cnt_col"]], 50)
    x90 = _percentile_count(df[panel["x_cnt_col"]], 90)

    prediction_rows = []
    contrast_rows = []

    for modifier_percentile in (50, 90):
        m_value = _percentile_count(df[panel["m_cnt_col"]], modifier_percentile)

        probs = {}
        for focal_percentile, focal_count in ((50, x50), (90, x90)):
            probability = _predict_probability_single(panel, focal_count, m_value)
            probs[focal_percentile] = probability
            prediction_rows.append({
                "panel": panel_letter,
                "table3_column": model_label,
                "model_key": panel["key"],
                "focal_exposure": focal_label,
                "focal_exposure_percentile": focal_percentile,
                "focal_exposure_count": focal_count,
                "interacting_exposure_held_constant": modifier_label,
                "interacting_exposure_percentile": modifier_percentile,
                "interacting_exposure_count": m_value,
                "predicted_reply_probability": probability,
                "predicted_reply_probability_pct": 100.0 * probability,
            })

        contrast_rows.append({
            "panel": panel_letter,
            "table3_column": model_label,
            "model_key": panel["key"],
            "focal_exposure": focal_label,
            "interacting_exposure_held_constant": modifier_label,
            "interacting_exposure_percentile": modifier_percentile,
            "interacting_exposure_count": m_value,
            "focal_exposure_p50_count": x50,
            "focal_exposure_p90_count": x90,
            "predicted_reply_probability_at_focal_p50": probs[50],
            "predicted_reply_probability_at_focal_p90": probs[90],
            "predicted_reply_probability_at_focal_p50_pct": 100.0 * probs[50],
            "predicted_reply_probability_at_focal_p90_pct": 100.0 * probs[90],
            "absolute_difference_percentage_points": 100.0 * (probs[90] - probs[50]),
            "prediction_framework": "full model.predict rows",
        })

    return prediction_rows, contrast_rows


absolute_profile_rows = []
absolute_prediction_rows = []
absolute_contrast_rows = []

for panel_letter, panel, model_label, focal_label, modifier_label in ABSOLUTE_PANEL_SPECS:
    for record in panel["profile_records"]:
        basis = record["basis"]
        if record["covariate"] == panel["x_log_name"]:
            basis = "contrasted at observed p50 and p90"
        elif record["covariate"] == panel["m_log_name"]:
            basis = "held at observed p50 or p90"

        absolute_profile_rows.append({
            "panel": panel_letter,
            "table3_column": model_label,
            "model_key": panel["key"],
            "covariate": record["covariate"],
            "value": record["display_value"],
            "basis": basis,
        })

    pred_rows, contrast_rows = _absolute_contrasts(
        panel,
        panel_letter,
        model_label,
        focal_label,
        modifier_label,
    )
    absolute_prediction_rows.extend(pred_rows)
    absolute_contrast_rows.extend(contrast_rows)

absolute_representative_covariates = pd.DataFrame(absolute_profile_rows)
absolute_predicted_probabilities = pd.DataFrame(absolute_prediction_rows)
absolute_absolute_probability_contrasts = pd.DataFrame(absolute_contrast_rows)


# ----------------------------
# Outputs
# ----------------------------
ABSOLUTE_OUTPUT_DIR = (
    Path(TABLE_DIR)
    if "TABLE_DIR" in globals()
    else (
        Path(RUN_DIR) / "tables"
        if "RUN_DIR" in globals()
        else Path.cwd()
    )
)
ABSOLUTE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABSOLUTE_PROFILE_PATH = ABSOLUTE_OUTPUT_DIR / "absolute_representative_covariate_values_table3_30d.csv"
ABSOLUTE_PREDICTION_PATH = ABSOLUTE_OUTPUT_DIR / "absolute_corrected_predicted_probabilities_table3_30d.csv"
ABSOLUTE_CONTRAST_PATH = ABSOLUTE_OUTPUT_DIR / "absolute_absolute_predicted_probability_contrasts_table3_30d.csv"

absolute_representative_covariates.to_csv(ABSOLUTE_PROFILE_PATH, index=False)
absolute_predicted_probabilities.to_csv(ABSOLUTE_PREDICTION_PATH, index=False)
absolute_absolute_probability_contrasts.to_csv(ABSOLUTE_CONTRAST_PATH, index=False)


# ----------------------------
# Manuscript-facing summary
# ----------------------------
absolute_summary_for_reporting = (
    absolute_absolute_probability_contrasts
    .assign(
        focal_p50_count=lambda d: d["focal_exposure_p50_count"].map(lambda x: f"{x:.1f}"),
        focal_p90_count=lambda d: d["focal_exposure_p90_count"].map(lambda x: f"{x:.1f}"),
        held_count=lambda d: d["interacting_exposure_count"].map(lambda x: f"{x:.1f}"),
        predicted_probability_p50=lambda d: d[
            "predicted_reply_probability_at_focal_p50_pct"
        ].map(lambda x: f"{x:.1f}%"),
        predicted_probability_p90=lambda d: d[
            "predicted_reply_probability_at_focal_p90_pct"
        ].map(lambda x: f"{x:.1f}%"),
        absolute_difference=lambda d: d[
            "absolute_difference_percentage_points"
        ].map(lambda x: f"{x:+.1f} pp"),
    )
    [[
        "panel",
        "table3_column",
        "focal_exposure",
        "focal_p50_count",
        "focal_p90_count",
        "interacting_exposure_held_constant",
        "interacting_exposure_percentile",
        "held_count",
        "predicted_probability_p50",
        "predicted_probability_p90",
        "absolute_difference",
    ]]
)

print("Representative covariate values:")
display(absolute_representative_covariates)

print("Corrected predicted probabilities:")
display(absolute_predicted_probabilities)

print("Absolute percentage-point contrasts:")
display(absolute_summary_for_reporting)

print("Saved representative covariates:", ABSOLUTE_PROFILE_PATH)
print("Saved corrected predicted probabilities:", ABSOLUTE_PREDICTION_PATH)
print("Saved absolute contrasts:", ABSOLUTE_CONTRAST_PATH)


## 11. Table S8: Construction and Distribution of the Recent Healthcare Encounter Covariate

This table describes the 7-day recent-encounter covariate included in the primary encounter-adjusted models. Encounter-type percentages use encounter instances as the denominator because the type-specific fields are encounter counts rather than mutually exclusive thread characteristics.

In [ ]:
# Table S8. Recent healthcare encounters in the primary HCW-initiated thread analysis
# This cell creates descriptive output only; it does not refit any model.

from IPython.display import HTML, display

TABLE_S8_TITLE = (
    "Table S8. Construction and Distribution of the Recent "
    "Healthcare Encounter Covariate"
)

TABLE_S8_NOTE = (
    "Values are based on HCW-initiated threads included in the primary encounter-adjusted "
    "analyses. The recent healthcare encounter covariate equals the number of documented "
    "healthcare encounters during the 7 days preceding thread initiation and enters the "
    "regression models as log(1 + count). Thread-level percentages use all included threads "
    "as the denominator. Encounter-type percentages use all encounter instances within the "
    "thread-specific 7-day look-back windows as the denominator; therefore, a single source "
    "encounter may contribute to more than one thread when look-back windows overlap. "
    "Hospital inpatient, hospital outpatient, and other hospital encounters are combined "
    "into the hospital category."
)

primary_keys = ["BASE_HCWxSYS_30d", "BASE_HCWxSYS_90d"]
missing_model_keys = [key for key in primary_keys if key not in MODELS_BASE]
if missing_model_keys:
    raise KeyError(
        "Run the primary encounter-adjusted model section before Table S8. "
        f"Missing model keys: {missing_model_keys}"
    )

primary_30d = MODELS_BASE["BASE_HCWxSYS_30d"]["df_fit"].copy()
primary_90d = MODELS_BASE["BASE_HCWxSYS_90d"]["df_fit"].copy()

# The primary 30- and 90-day models should use the same thread observations.
if len(primary_30d) != len(primary_90d) or not primary_30d.index.equals(primary_90d.index):
    raise ValueError(
        "The primary 30- and 90-day model samples differ. "
        "Create separate descriptive tables or reconcile model missingness before reporting Table S8."
    )

table_sample = primary_30d
required_encounter_cols = [
    "encounters_past_7d",
    "log_encounters_past_7d",
    "clinic_encounters_past_7d",
    "ed_encounters_past_7d",
    "hospital_encounters_past_7d",
    "other_encounters_past_7d",
]
missing_encounter_cols = [c for c in required_encounter_cols if c not in table_sample.columns]
if missing_encounter_cols:
    raise KeyError(f"Missing encounter columns required for Table S8: {missing_encounter_cols}")
if table_sample[required_encounter_cols].isna().any().any():
    raise ValueError("Encounter variables contain missing values in the primary model sample.")

encounter_counts = table_sample["encounters_past_7d"].astype(float)
if (encounter_counts < 0).any():
    raise ValueError("encounters_past_7d contains negative values.")

type_columns = {
    "Outpatient clinic": "clinic_encounters_past_7d",
    "Emergency department": "ed_encounters_past_7d",
    "Hospital": "hospital_encounters_past_7d",
    "Other": "other_encounters_past_7d",
}
type_counts = {
    label: int(table_sample[column].astype(float).sum())
    for label, column in type_columns.items()
}
total_encounter_instances = int(encounter_counts.sum())
if sum(type_counts.values()) != total_encounter_instances:
    raise ValueError(
        "Type-specific encounter counts do not sum to encounters_past_7d. "
        "Review encounter grouping before reporting Table S8."
    )

n_threads = int(len(table_sample))
n_with_encounter = int(encounter_counts.gt(0).sum())
mean_encounters = float(encounter_counts.mean())
sd_encounters = float(encounter_counts.std(ddof=1))
q1, median, q3 = encounter_counts.quantile([0.25, 0.50, 0.75]).tolist()
minimum = int(encounter_counts.min())
maximum = int(encounter_counts.max())
p90, p95, p99 = encounter_counts.quantile([0.90, 0.95, 0.99]).tolist()
distribution_counts = {
    "Threads with 0 prior encounters, n (%)": int(encounter_counts.eq(0).sum()),
    "Threads with 1 prior encounter, n (%)": int(encounter_counts.eq(1).sum()),
    "Threads with 2 prior encounters, n (%)": int(encounter_counts.eq(2).sum()),
    "Threads with 3 prior encounters, n (%)": int(encounter_counts.eq(3).sum()),
    "Threads with 4 or more prior encounters, n (%)": int(encounter_counts.ge(4).sum()),
}
if sum(distribution_counts.values()) != n_threads:
    raise ValueError("Encounter-count distribution categories do not sum to the thread count.")
rounded_type_percentage_sum = sum(
    round(100 * count / total_encounter_instances, 1)
    for count in type_counts.values()
) if total_encounter_instances else 0.0
if total_encounter_instances and not np.isclose(rounded_type_percentage_sum, 100.0, atol=0.2):
    raise ValueError("Rounded encounter-type percentages do not sum to approximately 100%.")

def format_n_pct(n, denominator):
    pct = 100 * n / denominator if denominator else 0.0
    return f"{n:,} ({pct:.1f}%)"

table_s8_rows = [
    {"Characteristic": "HCW-initiated threads, N", "Value": f"{n_threads:,}"},
    {
        "Characteristic": "Threads with \u22651 encounter during the previous 7 days, n (%)",
        "Value": format_n_pct(n_with_encounter, n_threads),
    },
    {
        "Characteristic": "Number of encounters during the previous 7 days, mean (SD)",
        "Value": f"{mean_encounters:.2f} ({sd_encounters:.2f})",
    },
    {
        "Characteristic": "Number of encounters during the previous 7 days, median (IQR)",
        "Value": f"{median:.0f} ({q1:.0f}\u2013{q3:.0f})",
    },
    {"Characteristic": "Distribution of prior encounter counts", "Value": ""},
]
table_s8_rows.extend(
    {"Characteristic": f"    {label}", "Value": format_n_pct(count, n_threads)}
    for label, count in distribution_counts.items()
)
table_s8_rows.extend([
    {"Characteristic": "    Minimum", "Value": f"{minimum:,}"},
    {"Characteristic": "    Maximum", "Value": f"{maximum:,}"},
    {"Characteristic": "    90th percentile", "Value": f"{p90:.0f}"},
    {"Characteristic": "    95th percentile", "Value": f"{p95:.0f}"},
    {"Characteristic": "    99th percentile", "Value": f"{p99:.0f}"},
    {
        "Characteristic": "Encounter instances contributing across 7-day windows, n",
        "Value": f"{total_encounter_instances:,}",
    },
    {"Characteristic": "Encounter type contributing to the covariate, n (%)", "Value": ""},
])
table_s8_rows.extend(
    {
        "Characteristic": f"    {label}",
        "Value": format_n_pct(count, total_encounter_instances),
    }
    for label, count in type_counts.items()
)

table_s8 = pd.DataFrame(table_s8_rows)

def style_table_s8_row(row):
    if row["Value"] == "":
        return ["font-weight: bold; background-color: #f2f2f2;"] * len(row)
    return [""] * len(row)

table_s8_styled = (
    table_s8.style
    .hide(axis="index")
    .apply(style_table_s8_row, axis=1)
    .set_properties(subset=["Characteristic"], **{"text-align": "left"})
    .set_properties(subset=["Value"], **{"text-align": "right"})
    .set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        {"selector": "th", "props": [("border-top", "1.5px solid black"), ("border-bottom", "1px solid black"), ("padding", "6px")]},
        {"selector": "td", "props": [("border", "none"), ("padding", "6px")]},
        {"selector": "tbody tr:last-child td", "props": [("border-bottom", "1.5px solid black")]},
    ])
)

display(HTML(f"<p><strong>{TABLE_S8_TITLE}</strong></p>"))
display(table_s8_styled)
display(HTML(f"<p><em>Note.</em> {TABLE_S8_NOTE}</p>"))

table_s8_csv_path = TABLE_DIR / "table_S8_recent_healthcare_encounters.csv"
table_s8_html_path = TABLE_DIR / "table_S8_recent_healthcare_encounters.html"
table_s8.to_csv(table_s8_csv_path, index=False)
table_s8_html = (
    f"<p><strong>{TABLE_S8_TITLE}</strong></p>"
    + table_s8_styled.to_html()
    + f"<p><em>Note.</em> {TABLE_S8_NOTE}</p>"
)
table_s8_html_path.write_text(table_s8_html, encoding="utf-8")

print("Saved:", table_s8_csv_path)
print("Saved:", table_s8_html_path)


## Aggregate model handoff

Export the fitted primary models and their full patient-clustered covariance matrices for notebook 10. This writes aggregate model summaries and representative prediction settings; no patient-level rows are exported.


In [ ]:
import sys
if str(REPO_ROOT) not in sys.path:sys.path.insert(0,str(REPO_ROOT))
from analysis.model_exports import export_primary
PRIMARY_AGGREGATES=export_primary({"A":panelA_30d,"B":panelB_30d,"C":panelC_30d},RUN_DIR)
print(PRIMARY_AGGREGATES)
